# Robustness studies — pre-processing and partial observability

Paper-ready figures and tables for the two sensitivity studies of the revision:

* **Study A — pre-processing.** What the Louvain step (resolution sweep, or no Louvain at all)
  and hub removal do to the graph, to the laundering patterns, and to the GARG-AML score
  (code task 4).
* **Study B — limited view.** The score computed from what one institution can see — a single
  bank (`012`) and the 50 largest banks pooled (`top50`) — against the same clients on the
  full network (code task 5).

This notebook **recomputes nothing expensive**. It digests files the pipeline already wrote
and turns them into assets you can drop into the manuscript:

| output | where |
| --- | --- |
| figures | `<results dir>/paper_robustness/fig_*.pdf` (vector, embedded fonts) |
| tables | `<results dir>/paper_robustness/table_*.tex` (booktabs) with a `.csv` twin of each |

The results directory is `GARGAML_RESULTS_DIR`, defaulting here to `results-revision/`; a few
older inputs that only exist in `results/` are found there (the inventory below says which).

**Where the inputs come from**

```bash
python scripts/gargaml_undirected.py && python scripts/gargaml_directed.py   # block measures, per arm
GARGAML_DATASET=louvain_sweep python scripts/distribution_scores.py          # pure-score metrics per arm
python scripts/pattern_splitting.py                                          # patterns vs Louvain
python scripts/partial_observability.py                                      # study B
```

### Five things to keep in mind when reading the output

1. **Study A compares the pure GARG-AML score only**, directed beside undirected — no tree,
   boosting or GraphSAGE model is fitted on a sweep arm. The published setting (`r = 10`) is read
   from its *full-population* row, never the fold mean, so every column is the same kind of number.
2. **The pure score is massively tied**, so `P@K` / `R@K` / `lift@K` of the sweep are tie-breaking
   artefacts and are **not reported** for study A. The ties table shows why. Study B reports them
   beside `ties@K`.
3. **Hub-removal arms are scored on the accounts that remain.** The *k* deleted hubs have no score, so the
   evaluated population is *k* accounts smaller, and some of the deleted hubs carry laundering transactions, so the
   arms also have slightly fewer positives. The tables say by how much.
4. **Study B's `full` column is not Tables 10–11.** Louvain is off on both sides of that comparison
   (otherwise the two regimes would be reduced by different partitions), so it is a baseline built
   for this comparison.
5. **Directed vs undirected at one resolution is not purely a direction effect.** `community_map`
   runs Louvain on `G.to_undirected()`, so the two directions can be reduced by different
   partitions at the same resolution (see `CLAUDE.md`, task 4).

In [ ]:
import os
import sys

# Run from the notebook directory; chdir to the repo root so the relative
# paths used elsewhere in the codebase work.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

# The revision runs land in results-revision/; a bare local run defaults to results/.
os.environ.setdefault("GARGAML_RESULTS_DIR", "results-revision")
print("cwd:", REPO_ROOT)

In [ ]:
import re
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from scipy.stats import spearmanr

from src.data.bank_views import view_name
from src.utils.evaluation import HEADLINE_CUTOFFS
from src.utils.graph_processing import DEFAULT_RESOLUTION
from src.utils.reporting import (HEADLINE_TARGETS, _resolution_order, format_cell,
                                 load_metrics, louvain_setting, pattern_splitting_table,
                                 severance_pattern_latex, sweep_table, write_table)
from src.utils.runtime import resolve_results_dir

warnings.filterwarnings("ignore", category=FutureWarning)

RESULTS_DIR = Path(resolve_results_dir())
FALLBACK_DIR = Path("results")          # a few older inputs only exist here
OUT = RESULTS_DIR / "paper_robustness"
OUT.mkdir(parents=True, exist_ok=True)

DATASET = "HI-Small"
DIRECTIONS = ["undirected", "directed"]
INSTITUTIONS = ["012", "top50"]         # must match scripts/partial_observability.py
SCORE_THRESHOLD = 0.5                   # same rule as scripts/distribution_scores.py
MIN_POSITIVES = 30                      # below this a rank metric is shown but not interpreted

LOUVAIN_ARMS = [DATASET, f"{DATASET}_nolouvain"] + [f"{DATASET}_res{r}" for r in (1, 5, 20, 50)]
HUB_ARMS = [f"{DATASET}_hubs{k}" for k in (5, 10, 100)]

print("results dir :", RESULTS_DIR.resolve())
print("output dir  :", OUT.resolve())

In [ ]:
# ---- Figure style -----------------------------------------------------------------
# Reference palette (slots 1-3 validate together on every pair). Colour follows the
# entity: blue/orange always mean undirected/directed. Every series also carries a
# marker or line style, so nothing relies on colour alone (print, greyscale, CVD).
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3de"
DIR_COLOUR = {"undirected": "#2a78d6", "directed": "#eb6834"}
DIR_MARKER = {"undirected": "o", "directed": "s"}
DIR_STYLE = {"undirected": "-", "directed": "--"}
DIR_LABEL = {"undirected": "Undirected", "directed": "Directed"}
NEUTRAL = "#8a8984"
LAUNDERING = "#4a3aa7"
PATTERN_COLOUR = {"GATHER-SCATTER": "#4a3aa7", "SCATTER-GATHER": "#1baf7a", "ALL": INK_2}
PATTERN_MARKER = {"GATHER-SCATTER": "o", "SCATTER-GATHER": "s", "ALL": "^"}
TARGET_SHORT = {"Is Laundering": "Any laundering", "GATHER-SCATTER": "Gather-scatter",
                "SCATTER-GATHER": "Scatter-gather"}
SEQ = LinearSegmentedColormap.from_list("seq", ["#eaf1fb", "#2a78d6", "#123f7a"])
DIV = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])

plt.rcParams.update({
    "figure.dpi": 110, "savefig.bbox": "tight", "pdf.fonttype": 42, "font.size": 9,
    "text.color": INK, "axes.labelcolor": INK, "axes.edgecolor": INK_2,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 9, "axes.titleweight": "regular", "legend.frameon": False,
    "legend.fontsize": 8,
})

# ---- Helpers ----------------------------------------------------------------------
WRITTEN = []
TABLES = {}                              # stem -> the formatted table, for the final section
(OUT / "preview").mkdir(exist_ok=True)   # PNG previews of the figures; never part of the paper assets


def find(name):
    # The first results directory that holds the file, or None.
    for d in (RESULTS_DIR, FALLBACK_DIR):
        if (d / name).exists():
            return d / name
    return None


def save_fig(fig, stem):
    path = OUT / f"fig_{stem}.pdf"
    fig.savefig(path)
    fig.savefig(OUT / "preview" / f"fig_{stem}.png", dpi=150)
    WRITTEN.append(path)
    print("wrote", path)


def save_table(table, stem, caption, label, note=None, wide=None, column_format=None, fit=False):
    # fit=True shrinks the tabular to the float's width (needs graphicx in the manuscript). It is opt-in:
    # \resizebox would also enlarge a table that already fits.
    paths = write_table(table, stem, caption=caption, label=label, results_dir=str(OUT),
                        note=note, wide=wide, column_format=column_format)
    TABLES[stem] = table
    if not paths:
        print(f"(table_{stem}: no rows on disk, not written)")
    elif fit:
        tex = Path(paths[0])
        text = tex.read_text()
        text = text.replace("\\begin{tabular}", "\\resizebox{\\linewidth}{!}{%\n\\begin{tabular}", 1)
        text = text.replace("\\end{tabular}", "\\end{tabular}}", 1)
        tex.write_text(text)
    if paths:
        # reporting.to_latex puts the note straight after \end{tabular}; on a table narrower than the float the
        # note then starts beside the table. A paragraph break keeps it underneath.
        tex = Path(paths[0])
        tex.write_text(re.sub(r"(\\end\{tabular\}\}?)\n\\footnotesize", lambda m: m.group(1) + "\n\\par\\footnotesize",
                              tex.read_text()))
    for p in paths:
        WRITTEN.append(Path(p))
        print("wrote", p)
    return paths


def fmt(v, digits=3, thin=False):
    # "$0.123$" or "--"; the same cell format as every other table_*.tex.
    # thin=True stars a value that rests on too few positives to be interpreted.
    if v is None or pd.isna(v):
        return "--"
    text = format_cell(float(v), digits=digits, basis="single")
    return text[:-1] + "^{*}$" if thin else text


def num(v):
    return "--" if pd.isna(v) else f"{int(round(v)):,}".replace(",", "\\,")


def strip_tex(text):
    # LaTeX cell or header -> readable text (\shortstack{a\\b} becomes "a b")
    text = re.sub(r"\\shortstack\{(.*?)\}", lambda m: m.group(1).replace("\\\\", " "), str(text))
    return text.replace("$", "").replace("\\", "")


def show(table):
    # Display a LaTeX-formatted table without the markup.
    t = table.copy()
    for c in t.columns:
        t[c] = t[c].astype(str).map(strip_tex)
    t.columns = [strip_tex(c) for c in t.columns]
    display(t)


def slug(text):
    return str(text).lower().replace(" ", "-").replace(">=", "ge").replace(".", "")

In [ ]:
# ---- Inventory: what is on disk for each study ---------------------------------------
INVENTORY = [
    ("A", "Edges removed per setting, incl. laundering edges", ["preprocessing_severance.csv"]),
    ("A", "Edges removed, graph level (both datasets)", ["louvain_severance.csv"]),
    ("A", "Pattern splitting by resolution", ["pattern_splitting_summary.csv"]),
    ("A", "Pure score, published setting (r=10)", [f"{DATASET}_undirected_base_metrics.csv",
                                                  f"{DATASET}_directed_base_metrics.csv"]),
    ("A", "Pure score, Louvain sweep arms", [f"{DATASET}_res*_base_metrics.csv",
                                             f"{DATASET}_nolouvain_*_base_metrics.csv"]),
    ("A", "Pure score, hub-removal arms", [f"{DATASET}_hubs*_base_metrics.csv"]),
    ("A", "Block measures, hub-removal arms", [f"{DATASET}_hubs*_GARGAML_*.csv"]),
    ("A", "Stage-1 runtime per setting", [f"timing/{DATASET}_{d}_*.csv" for d in DIRECTIONS]
                                         + [f"timing/{DATASET}_{a}_*.csv" for a in ("res*", "nolouvain", "hubs*")]),
    ("B", "Coverage of each institution", ["appendix_coverage.csv"]),
    ("B", "Per-client scores, both regimes", [f"{DATASET}_bank*_partial_observability_accounts.csv"]),
    ("B", "Tidy metrics, both regimes", [f"{DATASET}_bank*_partial_observability_metrics.csv"]),
]

rows = []
for study, what, patterns in INVENTORY:
    hits = []
    for d in (RESULTS_DIR, FALLBACK_DIR):
        for pat in patterns:
            hits += [p for p in d.glob(pat)]
    # a file present in both directories counts once, from the primary
    seen, unique = set(), []
    for p in hits:
        if p.name not in seen:
            seen.add(p.name)
            unique.append(p)
    newest = max((datetime.fromtimestamp(p.stat().st_mtime) for p in unique), default=None)
    rows.append(dict(study=study, input=what, files=len(unique),
                     location=", ".join(sorted({p.parent.name for p in unique})) or "-",
                     newest=newest.strftime("%Y-%m-%d") if newest else "-",
                     status="ok" if unique else "PENDING"))
inventory = pd.DataFrame(rows)
display(inventory)
if (inventory["status"] == "PENDING").any():
    print("PENDING inputs are skipped below and reported where they would appear.")

---
# Study A — Louvain pre-processing and hub removal

Four questions, in the order the paper has to answer them:

1. **What does the pre-processing cost the graph?** Edges removed — and, because laundering edges
   are what the score is looking for, how many of *those* are removed.
2. **What does it cost the patterns?** Louvain splits patterns across communities; some lose the
   source → mule → target path the score needs.
3. **Does the choice move the score?** AUC-ROC / AUC-PR of the pure score across the sweep.
4. **What does it cost in computation time?** Wall-clock time of the score per setting, beside the
   ego-network sizes that drive it (A.6).

## A.1 What the pre-processing removes

`preprocessing_severance.csv` counts, per setting, the edges discarded and how many of them belong
to laundering attempts (undirected account-pair edges of `HI-Small`).

**Enrichment** is the share of laundering edges removed divided by the share of all edges removed:
1 means the setting removes laundering edges at the same rate as any other; above 1 it removes
them preferentially. A setting that removes few edges can still be costly if it is enriched.

In [ ]:
ARM_ORDER = ["hubs5", "hubs10", "hubs100", "r=1", "r=5", "r=10", "r=20", "r=50"]


def arm_long(key):
    # tick / legend text
    if key == "off":
        return "no Louvain"
    if key.startswith("hubs"):
        return f"top-{key[4:]} hubs"
    r = float(key[2:])
    return f"r = {r:g}" + (" (published)" if r == DEFAULT_RESOLUTION else "")


def arm_two_line(key):
    # tick text that wraps instead of colliding on a wide bar / heatmap axis
    if key.startswith("hubs"):
        return f"top-{key[4:]}\nhubs"
    return arm_long(key).replace(" (published)", "\n(published)")


def arm_short(key):
    # compact tick for small multiples; the dagger marks the published setting
    if key == "off":
        return "none"
    if key.startswith("hubs"):
        return f"h{key[4:]}"
    r = float(key[2:])
    return f"{r:g}" + ("†" if r == DEFAULT_RESOLUTION else "")


def arm_tex(key):
    if key == "off":
        return "No Louvain"
    if key.startswith("hubs"):
        return f"Hub removal, top {key[4:]}"
    r = float(key[2:])
    return f"Louvain $r{{=}}{r:g}$" + (" (published)" if r == DEFAULT_RESOLUTION else "")


sev_path = find("preprocessing_severance.csv")
sev = pd.read_csv(sev_path) if sev_path else pd.DataFrame()
print("read", sev_path)

overall = (sev[(sev["dataset"] == DATASET) & (sev["pattern"] == "ALL")]
           .set_index("setting").reindex(ARM_ORDER))
overall["enrichment"] = overall["pct_laundering_removed"] / overall["pct_removed"]
display(overall[["edges", "edges_removed", "pct_removed", "laundering_edges",
                 "laundering_removed", "pct_laundering_removed", "enrichment",
                 "pct_of_removed_laundering"]].round(2))

In [ ]:
# Table A1: the headline severance table.
rows = {arm_tex("off"): ["$0.0$", "$0.0$", "--", "--"]}
for arm in ARM_ORDER:
    r = overall.loc[arm]
    rows[arm_tex(arm)] = [fmt(r["pct_removed"], 1), fmt(r["pct_laundering_removed"], 1),
                          fmt(r["enrichment"], 2), fmt(r["pct_of_removed_laundering"], 2)]
tbl_sev = pd.DataFrame.from_dict(rows, orient="index",
                                 columns=["Edges removed (\\%)", "Laundering edges removed (\\%)",
                                          "Enrichment", "Laundering among removed (\\%)"])
tbl_sev.index.name = "Setting"
show(tbl_sev)

n_laundering = int(overall["laundering_edges"].iloc[0])
save_table(
    tbl_sev, f"prep_severance_{DATASET}",
    caption=(f"Edges discarded by each pre-processing setting on {DATASET} ({int(overall['edges'].iloc[0]):,} "
             f"account-pair edges, of which {n_laundering:,} belong to laundering attempts). Enrichment is the "
             "share of laundering edges removed divided by the share of all edges removed: 1 means laundering "
             "edges are removed at the same rate as any other edge."),
    label="tab:prep-severance",
    note=("Hub removal deletes the $k$ highest-degree accounts and runs no community detection; "
          "$r$ is the Louvain resolution."), fit=True)

In [ ]:
# Figure A1: edges removed, all vs laundering-attempt edges.
xpos = np.array([0, 1, 2, 3.6, 4.6, 5.6, 6.6, 7.6])
w = 0.36
fig, ax = plt.subplots(figsize=(7.0, 3.1))
ax.bar(xpos - w / 2, overall["pct_removed"], w, color=NEUTRAL, edgecolor="white", linewidth=1.2,
       label="all edges")
ax.bar(xpos + w / 2, overall["pct_laundering_removed"], w, color=LAUNDERING, edgecolor="white",
       linewidth=1.2, label="laundering-attempt edges")
for x, a, b in zip(xpos, overall["pct_removed"], overall["pct_laundering_removed"]):
    ax.text(x - w / 2, a + 0.6, f"{a:.1f}", ha="center", va="bottom", fontsize=7, color=INK_2)
    ax.text(x + w / 2, b + 0.6, f"{b:.1f}", ha="center", va="bottom", fontsize=7, color=INK_2)
ax.set_xticks(xpos)
ax.set_xticklabels([arm_two_line(a) for a in ARM_ORDER], fontsize=7.5)
ax.set_ylabel("edges removed (%)")
ax.set_ylim(0, max(overall["pct_laundering_removed"].max(), overall["pct_removed"].max()) * 1.15)
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left")
plt.tight_layout()
save_fig(fig, "prep_severance")
plt.show()

In [ ]:
# Figure A2 + Table A2: which laundering patterns lose edges.
PATTERN_ORDER = ["ALL", "GATHER-SCATTER", "SCATTER-GATHER", "FAN-OUT", "FAN-IN", "CYCLE",
                 "BIPARTITE", "STACK", "RANDOM", "Not Classified"]
pct = (sev[sev["dataset"] == DATASET]
       .pivot_table(index="pattern", columns="setting", values="pct_laundering_removed")
       .reindex(index=[p for p in PATTERN_ORDER if p in set(sev["pattern"])], columns=ARM_ORDER))
reference = overall["pct_removed"].to_frame().T.rename(index={"pct_removed": "All graph edges"})
data = pd.concat([reference, pct])


def heatmap(ax, data, cmap, vmin, vmax, text, is_dark, row_labels, col_labels, faded=()):
    values = data.values.astype(float)
    coloured = values.copy()
    if len(faded):
        coloured[list(faded), :] = np.nan            # shown as text only, not coloured
    mesh = ax.pcolormesh(coloured, cmap=cmap, vmin=vmin, vmax=vmax, edgecolors="white", linewidth=1.5)
    ax.set_xlim(0, values.shape[1])
    ax.set_ylim(values.shape[0], 0)
    ax.set_xticks(np.arange(values.shape[1]) + 0.5)
    ax.set_xticklabels(col_labels, fontsize=7.5)
    ax.set_yticks(np.arange(values.shape[0]) + 0.5)
    ax.set_yticklabels(row_labels, fontsize=7.5)
    ax.grid(False)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            v = values[i, j]
            ax.text(j + 0.5, i + 0.5, "–" if np.isnan(v) else text(v), ha="center",
                    va="center", fontsize=7, style="italic" if i in faded else "normal",
                    color=NEUTRAL if i in faded else ("white" if (not np.isnan(v) and is_dark(v)) else INK))
    return mesh


fig, ax = plt.subplots(figsize=(7.0, 3.6))
mesh = heatmap(ax, data, SEQ, 0, np.nanmax(data.values), lambda v: f"{v:.1f}",
               lambda v: v > 0.55 * np.nanmax(data.values), list(data.index),
               [arm_two_line(a) for a in ARM_ORDER])
ax.axhline(1, color="white", linewidth=4)          # set the graph-wide row apart
cb = fig.colorbar(mesh, ax=ax, fraction=0.03, pad=0.02)
cb.set_label("edges removed (%)", fontsize=8)
cb.outline.set_visible(False)
plt.tight_layout()
save_fig(fig, "prep_severance_patterns")
plt.show()

# The same numbers as a booktabs table (reporting.severance_pattern_latex), if you prefer it.
tex = severance_pattern_latex(sev, dataset=DATASET, label="tab:prep-severance-patterns")
path = OUT / f"table_prep_severance_patterns_{DATASET}.tex"
path.write_text(tex)
WRITTEN.append(path)
print("wrote", path)

The figure and the table above show the same numbers — pick one for the paper. The first row of the
figure (all graph edges) is the reference: a pattern whose cell is darker than that row is being
removed *more* than the graph as a whole.

## A.2 What it does to the laundering patterns

`pattern_splitting_summary.csv` follows each injected laundering attempt through the Louvain step:
how many communities its accounts land in, and whether its two-hop source → mule → target paths
survive. Two readings:

* **split** — the attempt's accounts span more than one community (some of its edges are cut);
* **destroyed** — the attempt had a two-hop path and no longer does, so the score has nothing left
  to find. One-hop patterns (fan-in, fan-out) have no two-hop structure to lose and read as gaps.

In [ ]:
split_path = find("pattern_splitting_summary.csv")
split = pd.read_csv(split_path)
print("read", split_path)
split["arm"] = [str(v) if str(v) == "off" or str(v).startswith("hubs") else f"r={float(v):g}"
                for v in split["resolution"]]
SPLIT_ORDER = ["off", "r=1", "r=5", "r=10", "r=20", "r=50"]
SPLIT_TICK = {"off": "none", "r=1": "1", "r=5": "5", "r=10": "10†", "r=20": "20", "r=50": "50"}

datasets = list(split["dataset"].unique())
fig, axes = plt.subplots(2, len(datasets), figsize=(3.6 * len(datasets), 4.6), sharex=True, squeeze=False)
for j, ds in enumerate(datasets):
    sub = split[split["dataset"] == ds]
    for i, (col, ylabel) in enumerate([("pct_split", "attempts split across\ncommunities (%)"),
                                       ("pct_destroyed", "attempts with no surviving\ntwo-hop path (%)")]):
        ax = axes[i, j]
        for pat in ["GATHER-SCATTER", "SCATTER-GATHER", "ALL"]:
            s = sub[sub["pattern_type"] == pat].set_index("arm").reindex(SPLIT_ORDER)
            ax.plot(range(len(SPLIT_ORDER)), s[col], color=PATTERN_COLOUR[pat], marker=PATTERN_MARKER[pat],
                    ms=4.5, lw=1.6, label=pat.title() if pat != "ALL" else "All attempts")
        ax.set_ylim(0, 100)
        if j == 0:
            ax.set_ylabel(ylabel)
        if i == 0:
            n = int(sub[sub["pattern_type"] == "ALL"]["attempts"].iloc[0])
            ax.set_title(f"{ds} ({n:,} attempts)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.04))
for ax in axes[1]:
    ax.set_xticks(range(len(SPLIT_ORDER)))
    ax.set_xticklabels([SPLIT_TICK[a] for a in SPLIT_ORDER])
fig.supxlabel("Louvain resolution r (none = no reduction; † published)", fontsize=8.5, y=0.0)
plt.tight_layout()
save_fig(fig, "prep_pattern_splitting")
plt.show()

In [ ]:
# Tables A3 / A4: the same summary, by pattern type, from reporting.pattern_splitting_table.
SPLIT_CAPTIONS = {
    "pct_destroyed": ("Percentage of laundering attempts left undetectable by the Louvain step: "
                      "the attempt had a source--mule--target path and no longer does. "
                      "One-hop patterns have no such path and are marked --."),
    "pct_split": ("Percentage of laundering attempts whose accounts span more than one Louvain "
                  "community, so that at least one of their edges is discarded."),
}
for column, caption in SPLIT_CAPTIONS.items():
    t = pattern_splitting_table(str(split_path.parent), column=column)
    if not t.empty:
        t.index.names = ["Dataset", "Pattern"]
        show(t)
    save_table(t, f"prep_splitting_{column}", caption=caption, label=f"tab:prep-splitting-{column}",
               note=("$r$ is the Louvain resolution and $r=10$ the published setting; hub removal deletes the $k$ "
                     "highest-degree accounts (HI-Small only)."
                     + (" Hub removal has no communities to be split across, so it is not applicable (--)."
                        if column == "pct_split" else "")),
               wide=True, fit=True)

## A.3 Does the choice move the score?

Pure GARG-AML score on the full `HI-Small` population, one arm per pre-processing setting, directed
beside undirected. **No model is fitted**, so there are no folds and no confidence intervals: every
point is a single full-population number. Because of that, differences of a few thousandths should be
read as noise-level — the question is whether *any* setting moves the result materially, not which
setting wins.

**Tie warning.** The score takes very few distinct values at the top of the ranking (see the ties table
below), so the precision / recall / lift *at K* of this sweep depend on how ties are broken, not on the
score. Only the threshold-free metrics — AUC-ROC and AUC-PR — are shown.

In [ ]:
raw = load_metrics(str(RESULTS_DIR), datasets=LOUVAIN_ARMS + HUB_ARMS)

score = raw[(raw["features"] == "score") & raw["model"].isin(["gargaml_u", "gargaml_d"])
            & raw["fold"].isna() & (raw["status"] == "ok")]
score = louvain_setting(score)
score = score[score["base_dataset"] == DATASET].copy()


def arm_key(resolution):
    # "off" / "hubs<k>" stay as they are; a numeric resolution becomes "r=<r>"
    return resolution if isinstance(resolution, str) else f"r={float(resolution):g}"


score["arm"] = [arm_key(v) for v in score["resolution"]]
ORDER = [arm_key(v) for v in _resolution_order(score["resolution"].unique())]
PUBLISHED = f"r={DEFAULT_RESOLUTION:g}"

print("arms on disk, in order of how much they reduce the graph:", ORDER)
missing_hubs = [a for a in HUB_ARMS if a.split("_")[-1] not in ORDER]
if missing_hubs:
    print("\nNo pure-score metrics yet for:", ", ".join(missing_hubs))
    print("Their block measures exist (HI-Small_hubs<k>_GARGAML_*.csv) and their edge severance is in A.1;")
    print("score metrics need an evaluation run on those arms (README: hub-removal arms -> scripts/gargaml_tree.py).")

In [ ]:
# Stage-1 runtime of every setting, for the runtime panel of the sweep figure; A.6 tabulates it and shows what drives it.
TIMING_DIR = RESULTS_DIR / "timing"
RT_ORDER = (["off"] + [a for a in ARM_ORDER if not a.startswith("hubs")]
            + [a for a in ARM_ORDER if a.startswith("hubs")])      # no Louvain, r = 1 ... 50, then the hub arms


def arm_dataset(key):
    # "off" / "r=<r>" / "hubs<k>" -> the dataset name the measure scripts ran under
    if key == "off":
        return f"{DATASET}_nolouvain"
    if key.startswith("hubs"):
        return f"{DATASET}_{key}"
    r = float(key[2:])
    return DATASET if r == DEFAULT_RESOLUTION else f"{DATASET}_res{r:g}"


key_of = {arm_dataset(a): a for a in RT_ORDER}
timing = pd.concat([pd.read_csv(p) for p in TIMING_DIR.glob(f"{DATASET}*.csv")], ignore_index=True)
timing = timing[timing["dataset"].isin(key_of)].copy()
# A resubmitted job leaves a second file; keep the latest, as VisualisationRunTime.ipynb does.
timing = timing.sort_values("timestamp").drop_duplicates(["dataset", "direction"], keep="last")
timing["arm"] = timing["dataset"].map(key_of)
n_workers = timing["n_workers"].unique()
assert len(n_workers) == 1, f"runs with different worker counts are not comparable: {n_workers}"
n_workers = int(n_workers[0])

seconds = timing.pivot(index="arm", columns="direction", values="seconds").reindex(index=RT_ORDER, columns=DIRECTIONS)
missing_rt = [f"{a} ({d})" for a in RT_ORDER for d in DIRECTIONS if pd.isna(seconds.loc[a, d])]
print(f"stage-1 runtime (s), {n_workers} workers" + (f"; no timing file for: {', '.join(missing_rt)}" if missing_rt else ""))
display(seconds)

In [ ]:
def wide_cells(direction, metric):
    # (target, cut-off) x arm, one full-population value per cell
    s = score[(score["direction"] == direction) & (score["metric"] == metric)
              & score["cutoff"].isin(HEADLINE_CUTOFFS) & score["target"].isin(HEADLINE_TARGETS)]
    wide = s.pivot_table(index=["target", "cutoff"], columns="arm", values="value", aggfunc="last")
    index = pd.MultiIndex.from_product([HEADLINE_TARGETS, HEADLINE_CUTOFFS], names=["target", "cutoff"])
    return wide.reindex(index=index, columns=ORDER)


cells = {m: {d: wide_cells(d, m) for d in DIRECTIONS} for m in ("AUC_ROC", "AUC_PR")}

# Positives behind each headline cell: the high cut-offs and the pattern targets rest on far fewer.
n_pos = (score[(score["metric"] == "AUC_ROC") & (score["direction"] == "undirected")
               & (score["arm"] == PUBLISHED) & score["cutoff"].isin(HEADLINE_CUTOFFS)
               & score["target"].isin(HEADLINE_TARGETS)]
         .pivot_table(index="target", columns="cutoff", values="n_pos", aggfunc="last")
         .reindex(HEADLINE_TARGETS))
print("positives per (target, cut-off) cell, of", f"{int(score['n_test'].iloc[0]):,}", "scored accounts:")
display(n_pos.astype(int))

# A hub arm is scored on the accounts that remain after the hubs are deleted, so its population differs.
pop_by_arm = (score[(score["metric"] == "AUC_ROC") & (score["direction"] == "undirected")
                    & (score["target"] == "Is Laundering") & (score["cutoff"] == 0.0)]
              .drop_duplicates("arm", keep="last").set_index("arm")[["n_test", "n_pos"]].reindex(ORDER).astype(int))
lost_pos = pop_by_arm.loc[PUBLISHED, "n_pos"] - pop_by_arm["n_pos"]      # accounts with a laundering transaction removed
if any(a.startswith("hubs") for a in ORDER):
    print("\npopulation scored per arm (cut-off 0.0, any laundering):")
    display(pop_by_arm.assign(laundering_accounts_removed=lost_pos))

# Cells resting on too few positives are shown but not interpreted or summarised.
reliable = pd.Series({(t, c): bool(n_pos.loc[t, c] >= MIN_POSITIVES) for t in n_pos.index for c in n_pos.columns})
reliable.index.names = ["target", "cutoff"]
keep_rows = reliable.reindex(cells["AUC_ROC"]["undirected"].index).values
thin_cells = [f"{TARGET_SHORT[t].lower()} at {c:g}" for (t, c), ok in reliable.items() if not ok]
thin_note = (f"Cells resting on fewer than {MIN_POSITIVES} positives ({'; '.join(thin_cells)}) are not interpretable."
             if thin_cells else "")
print(f"\n{int(keep_rows.sum())} of {len(keep_rows)} cells have at least {MIN_POSITIVES} positives; "
      f"{thin_note or 'none are thin.'}")

# The interpretable cells are chosen on the published arm; check that no other arm drops below the floor in one of them.
npos_arm = (score[(score["metric"] == "AUC_ROC") & (score["direction"] == "undirected")
                  & score["cutoff"].isin(HEADLINE_CUTOFFS) & score["target"].isin(HEADLINE_TARGETS)]
            .pivot_table(index=["target", "cutoff"], columns="arm", values="n_pos", aggfunc="last")
            .reindex(index=cells["AUC_ROC"]["undirected"].index, columns=ORDER))
below = (npos_arm[keep_rows] < MIN_POSITIVES).any(axis=0)
print("arms with an interpretable cell below the floor:", list(below[below].index) or "none")


def panel(target, cutoff, direction, metric):
    s = score[(score["target"] == target) & (score["cutoff"] == cutoff)
              & (score["direction"] == direction) & (score["metric"] == metric)]
    return s.drop_duplicates("arm", keep="last").set_index("arm").reindex(ORDER)


LOU = [a for a in ORDER if not a.startswith("hubs")]
HUB = [a for a in ORDER if a.startswith("hubs")]
XPOS = {a: i for i, a in enumerate(LOU)}
XPOS.update({a: len(LOU) + 0.7 + i for i, a in enumerate(HUB)})


def plot_sweep(target, stem, runtime=False):
    # runtime=True adds a bottom panel across the cut-offs: the stage-1 runtime of each setting, seconds on every point.
    ncol = len(HEADLINE_CUTOFFS)
    if runtime:
        fig = plt.figure(figsize=(8.4, 6.4), layout="constrained")   # spaces each row on its own; tight_layout cannot
        gs = fig.add_gridspec(3, ncol, height_ratios=[1, 1, 1.2])
        axes = np.empty((2, ncol), dtype=object)
        for i in range(2):
            for j in range(ncol):
                axes[i, j] = fig.add_subplot(gs[i, j], sharex=None if i == j == 0 else axes[0, 0])
        for ax in axes[0]:
            ax.tick_params(labelbottom=False)
        rt_ax = fig.add_subplot(gs[2, :])
    else:
        fig, axes = plt.subplots(2, ncol, figsize=(8.4, 4.1), sharex=True)
    lou_x, hub_x = [XPOS[a] for a in LOU], [XPOS[a] for a in HUB]
    rocs = []
    for j, cutoff in enumerate(HEADLINE_CUTOFFS):
        for d in DIRECTIONS:
            roc, pr = panel(target, cutoff, d, "AUC_ROC"), panel(target, cutoff, d, "AUC_PR")
            rel = pr["value"] / (pr["n_pos"] / pr["n_test"])
            style = dict(color=DIR_COLOUR[d], marker=DIR_MARKER[d], ls=DIR_STYLE[d], ms=4, lw=1.6)
            for ax, series in ((axes[0, j], roc["value"]), (axes[1, j], rel)):
                ax.plot(lou_x, series.loc[LOU], **style)        # one line through none and the resolutions
                if HUB:                                         # hub removal is a different treatment: markers only
                    ax.plot(hub_x, series.loc[HUB], **{**style, "ls": "none"})
            rocs += list(roc["value"].dropna())
        axes[0, j].axhline(0.5, color=INK_2, lw=0.8, ls=":")
        axes[1, j].axhline(1.0, color=INK_2, lw=0.8, ls=":")
        if HUB:
            for ax in axes[:, j]:
                ax.axvline(len(LOU) - 0.15, color=INK_2, lw=0.6, ls=(0, (2, 3)), alpha=0.6)
        n = int(n_pos.loc[target, cutoff])
        axes[0, j].set_title(f"cut-off {cutoff:g}\n{n:,} positives" + ("*" if n < MIN_POSITIVES else ""))
        if n < MIN_POSITIVES:
            for ax in axes[:, j]:
                ax.set_facecolor("#f0efec")
    shaded = any(n_pos.loc[target, c] < MIN_POSITIVES for c in HEADLINE_CUTOFFS)
    if rocs:
        for ax in axes[0]:
            ax.set_ylim(min(rocs) - 0.03, max(rocs) + 0.03)
    for ax in axes[0, 1:]:
        ax.tick_params(labelleft=False)
    axes[0, 0].set_ylabel("AUC-ROC")
    axes[1, 0].set_ylabel("AUC-PR / prevalence")
    for ax in axes[1]:
        ax.set_xticks([XPOS[a] for a in ORDER])
        ax.set_xticklabels([arm_short(a) if a in LOU else a[4:] for a in ORDER], fontsize=6.5)
        if HUB:
            for xs, text in ((lou_x, "Louvain r"), (hub_x, "top-k hubs")):
                ax.text(np.mean(xs), -0.30, text, transform=ax.get_xaxis_transform(), ha="center", va="top",
                        fontsize=6.5, color=INK_2)
    if runtime:
        for d in DIRECTIONS:
            style = dict(color=DIR_COLOUR[d], marker=DIR_MARKER[d], ls=DIR_STYLE[d], ms=4, lw=1.6)
            rt_ax.plot(lou_x, seconds.loc[LOU, d], **style)
            if HUB:
                rt_ax.plot(hub_x, seconds.loc[HUB, d], **{**style, "ls": "none"})
        for a in ORDER:
            pair = seconds.loc[a, DIRECTIONS].dropna()
            for d, v in pair.items():
                above = v == pair.max()   # the slower direction is labelled above its point, the faster below
                rt_ax.annotate(f"{v:,.0f}", (XPOS[a], v), xytext=(0, 4 if above else -4), textcoords="offset points",
                               ha="center", va="bottom" if above else "top", fontsize=6.5, color=INK_2)
        rt_ax.set_yscale("log")
        rt_ax.set_ylim(seconds.loc[ORDER].min().min() / 1.8, seconds.loc[ORDER].max().max() * 1.8)  # room for labels
        rt_ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
        rt_ax.set_ylabel("stage-1 runtime (s)")
        rt_ax.set_title("computation time per setting (the same at every cut-off; one run each)")
        rt_ax.set_xticks([XPOS[a] for a in ORDER])
        rt_ax.set_xticklabels([arm_short(a) if a in LOU else a[4:] for a in ORDER], fontsize=6.5)
        if HUB:
            rt_ax.axvline(len(LOU) - 0.15, color=INK_2, lw=0.6, ls=(0, (2, 3)), alpha=0.6)
            for xs, text in ((lou_x, "Louvain r"), (hub_x, "top-k hubs")):
                rt_ax.text(np.mean(xs), -0.16, text, transform=rt_ax.get_xaxis_transform(), ha="center",
                           va="top", fontsize=6.5, color=INK_2)
    handles = [Line2D([], [], color=DIR_COLOUR[d], marker=DIR_MARKER[d], ls=DIR_STYLE[d], ms=4,
                      lw=1.6, label=DIR_LABEL[d]) for d in DIRECTIONS]
    fig.legend(handles=handles, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.04))
    # Under constrained layout the label is placed below the runtime panel; the offsets fit the 4.1-inch figure.
    label_y = {} if runtime else {"y": (-0.10 if shaded else -0.06) if HUB else (-0.03 if shaded else -0.01)}
    fig.supxlabel("Louvain resolution r (none = no reduction; † published)"
                  + (" · number of hubs removed k, no Louvain" if HUB else "")
                  + (f"\n* shaded: fewer than {MIN_POSITIVES} positives, not interpretable" if shaded else ""),
                  fontsize=8.5, **label_y)
    if not runtime:
        plt.tight_layout()
    save_fig(fig, stem)
    plt.show()


plot_sweep("Is Laundering", "sweep_score_any-laundering", runtime=True)

*How to read it.* Top row: AUC-ROC, dotted line at 0.5 (random ranking). Middle row: AUC-PR divided by the
prevalence of positives in that cell, dotted line at 1 (random ranking) — it puts the four cut-offs, whose
prevalences differ by orders of magnitude, on one footing. Flat lines mean the pre-processing choice does
not move the score. Bottom panel: the stage-1 runtime of each setting, seconds on every point; it does not
depend on the cut-off, so one panel spans the four (A.6 has the table and its limits). The pattern-specific
versions follow without the runtime panel; they belong in the appendix.

In [ ]:
plot_sweep("GATHER-SCATTER", "sweep_score_gather-scatter")
plot_sweep("SCATTER-GATHER", "sweep_score_scatter-gather")

In [ ]:
# Figure A5: AUC-ROC change against the published setting, every headline cell.
def delta(direction, metric="AUC_ROC"):
    w = cells[metric][direction]
    return w.sub(w[PUBLISHED], axis=0).drop(columns=[PUBLISHED])


deltas = {d: delta(d) for d in DIRECTIONS}
vmax = max(np.nanmax(np.abs(d.values[keep_rows])) for d in deltas.values())
faded = list(np.where(~keep_rows)[0])
row_labels = [f"{TARGET_SHORT[t]} · {c:g}" + ("*" if not ok else "")
              for (t, c), ok in zip(deltas["undirected"].index, keep_rows)]
col_labels = [arm_long(a) for a in deltas["undirected"].columns]

fig, axes = plt.subplots(1, 2, figsize=(8.8, 4.4))
for ax, d in zip(axes, DIRECTIONS):
    mesh = heatmap(ax, deltas[d] * 100, DIV, -vmax * 100, vmax * 100, lambda v: f"{v:+.1f}",
                   lambda v: abs(v) > 0.7 * vmax * 100,
                   row_labels if d == "undirected" else [""] * len(row_labels), col_labels, faded=faded)
    for y in (4, 8):                                   # separate the three targets
        ax.axhline(y, color="white", linewidth=4)
    ax.set_title(DIR_LABEL[d])
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
cb = fig.colorbar(mesh, ax=axes, fraction=0.025, pad=0.02)
cb.set_label("AUC-ROC change vs r = 10 (points × 100)", fontsize=8)
cb.outline.set_visible(False)
if faded:
    fig.text(0.5, -0.08, f"* fewer than {MIN_POSITIVES} positives: values shown in grey, not coloured, not interpreted",
             ha="center", fontsize=8)
save_fig(fig, "sweep_score_delta_auc_roc")
plt.show()

In [ ]:
# Table A5: one line per setting - mean AUC-ROC over the twelve headline cells and the largest deviation.
rows = {}
for arm in ORDER:
    rec = {}
    for d in DIRECTIONS:
        w = cells["AUC_ROC"][d][keep_rows]
        rec[f"{DIR_LABEL[d]}: mean"] = fmt(w[arm].mean())
        rec[f"{DIR_LABEL[d]}: max $|\\Delta|$"] = "--" if arm == PUBLISHED else fmt((w[arm] - w[PUBLISHED]).abs().max())
    rows[arm_tex(arm)] = rec
tbl_summary = pd.DataFrame.from_dict(rows, orient="index")
tbl_summary.index.name = "Setting"
show(tbl_summary)
save_table(
    tbl_summary, f"sweep_score_summary_{DATASET}",
    caption=(f"AUC-ROC of the GARG-AML score on {DATASET} across pre-processing settings, directed beside "
             f"undirected. Each cell summarises the {int(keep_rows.sum())} headline (target, cut-off) cells with at least "
             f"{MIN_POSITIVES} positives: the mean AUC-ROC, and the largest absolute deviation from the published setting "
             "$r=10$. No model is fitted; every value is a full-population number."),
    label="tab:sweep-score-summary",
    note=("The headline cells are three targets (any laundering, gather-scatter, scatter-gather) by four label cut-offs. "
          + thin_note))

In [ ]:
# Tables A6 / A7: the full grid, as in the other table_sweep_score_* files.
for metric in ("AUC_ROC", "AUC_PR"):
    t = sweep_table(raw, DATASET, None, metric=metric, features="score", fold_mode="single")
    save_table(
        t, f"sweep_score_{DATASET}_{metric}",
        caption=(f"Sensitivity of the {metric.replace('_', '-')} of the GARG-AML score itself to the "
                 f"pre-processing, {DATASET}, directed beside undirected. No model is fitted: every column "
                 "is the score on the full population. Columns run from no reduction at all to the most "
                 "aggressive setting; $r=10$ is the published choice."),
        label=f"tab:sweep-score-{metric.lower().replace('_', '-')}",
        note=("Cells are deliberately not bolded: the question is whether the choice of resolution moves the "
              "result, not which resolution to select on the evaluation data. " + thin_note),
        wide=True, fit=True)

In [ ]:
# Table A8: why P@K is not reported - the tie group at the K-th position.
ties = score[(score["metric"] == "ties@K") & (score["K"] == 100) & (score["target"] == "Is Laundering")
             & (score["cutoff"] == 0.0)]
t = ties.pivot_table(index="direction", columns="arm", values="value", aggfunc="last").reindex(
    index=DIRECTIONS, columns=ORDER)
n_test = int(ties["n_test"].iloc[0])
tbl_ties = t.apply(lambda col: col.map(lambda v: "--" if pd.isna(v) else f"{int(v):,}".replace(",", "\\,")
                                       + f" ({100 * v / n_test:.0f}\\%)"))
tbl_ties.index = [DIR_LABEL[d] for d in tbl_ties.index]
tbl_ties.index.name = "Score"
tbl_ties.columns = [arm_long(a).replace(" (published)", "$^{\\dagger}$") for a in tbl_ties.columns]
show(tbl_ties)
save_table(
    tbl_ties, f"sweep_score_ties_{DATASET}",
    caption=(f"Number of accounts (share of the {n_test:,} scored) sharing the score of the 100th-ranked account. "
             "Wherever this exceeds 100, the top-100 is not uniquely defined and precision, recall and lift at K "
             "of the pure score depend on how ties are broken; they are therefore not reported for the sweep."),
    label="tab:sweep-score-ties",
    note="$^{\\dagger}$ published setting.", wide=True, fit=True)

## A.4 Downstream models across settings

`build_tables.py` writes a downstream table (decision tree / boosting) once at least two Louvain settings
have tree results on disk. The decision of 2026-10-10 was to run the sweep on the pure score only, so this
is expected to be empty; it is checked here so it appears automatically if that changes (for example once
the hub arms are evaluated).

In [ ]:
written_any = False
for d in DIRECTIONS:
    for metric in ("AUC_PR", "AUC_ROC"):
        t = sweep_table(raw, DATASET, d, metric=metric, features="full", n_folds=5)
        if t.empty:
            continue
        written_any = True
        save_table(
            t, f"sweep_models_{DATASET}_{d}_{metric}",
            caption=(f"Sensitivity of {metric.replace('_', '-')} to the pre-processing, {d} {DATASET}, with the "
                     "published feature configuration. Cells are fold means $\\pm$ one standard deviation."),
            label=f"tab:sweep-models-{d}-{metric.lower().replace('_', '-')}", wide=True)
if not written_any:
    print("No downstream-model sweep on disk (fewer than two settings have tree results) - nothing to write.")

## A.5 One-table summary for the main text

The ablation table the main text can carry on its own: what each setting **costs** (edges removed, laundering
edges removed, laundering attempts that lose every two-hop source → mule → target path) beside what it **does to
the score** (AUC-ROC, directed beside undirected). It reads the quantities computed above and adds nothing new.

Two limits to state when it is used. The AUC-ROC columns average the headline cells with enough positives, so they
summarise; the figure above shows the cells. And the hub-removal rows are scored on the accounts that remain after the
hubs are deleted, which are slightly fewer and carry slightly fewer positives than the Louvain rows (the note under the
table says by how many). A dash means "not computed for that setting".

In [ ]:
ABL_ORDER = ["off", "r=1", "r=5", "r=10", "r=20", "r=50", "hubs5", "hubs10", "hubs100"]
lost = (split[(split["dataset"] == DATASET) & (split["pattern_type"] == "ALL")]
        .set_index("arm")["pct_destroyed"])

rows = {}
for arm in ABL_ORDER:
    edges, laundering = ((0.0, 0.0) if arm == "off"
                         else (overall.loc[arm, "pct_removed"], overall.loc[arm, "pct_laundering_removed"]))
    rec = [fmt(edges, 1), fmt(laundering, 1), fmt(lost.get(arm, np.nan), 1)]
    for d in DIRECTIONS:
        w = cells["AUC_ROC"][d][keep_rows]
        rec.append(fmt(w[arm].mean()) if arm in w.columns else "--")
    rows[arm_tex(arm)] = rec

hub_arms = [a for a in ABL_ORDER if a.startswith("hubs")]
hub_note = ""
if all(a in ORDER for a in hub_arms):
    hub_note = ("Hub rows are scored on the accounts that remain after the hubs are deleted; removing the top "
                + " / ".join(a[4:] for a in hub_arms) + " hubs removes "
                + " / ".join(str(int(lost_pos[a])) for a in hub_arms) + " accounts with a laundering transaction. ")

tbl_abl = pd.DataFrame.from_dict(rows, orient="index", columns=[
    "\\shortstack{Edges\\\\removed (\\%)}", "\\shortstack{Laundering\\\\edges\\\\removed (\\%)}",
    "\\shortstack{Attempts losing\\\\every two-hop\\\\path (\\%)}",
    "\\shortstack{AUC-ROC\\\\undirected}", "\\shortstack{AUC-ROC\\\\directed}"])
tbl_abl.index.name = "Pre-processing"
show(tbl_abl)
save_table(
    tbl_abl, f"prep_ablation_{DATASET}",
    caption=(f"Ablation of the graph pre-processing on {DATASET}. The first three columns give what each setting costs: "
             "the share of all edges and of laundering-attempt edges it removes, and the share of laundering attempts "
             "that had a source--mule--target path and no longer do. The last two give the AUC-ROC of the GARG-AML score "
             f"itself, averaged over the {int(keep_rows.sum())} headline (target, cut-off) cells with at least "
             f"{MIN_POSITIVES} positives; no model is fitted and every value is a full-population number."),
    label="tab:prep-ablation",
    note=("Louvain $r=10$ is the published setting; hub removal deletes the $k$ highest-degree accounts and runs no "
          "community detection. "
          + ("A dash marks a quantity not computed for that setting. " if (tbl_abl == "--").any().any() else "")
          + hub_note
          + (f"Excluded from the AUC-ROC averages (fewer than {MIN_POSITIVES} positives): {'; '.join(thin_cells)}."
             if thin_cells else "")), wide=True)

In [ ]:
# ---- Study A: numbers for the text ------------------------------------------------
print("PRE-PROCESSING - numbers for the text\n")
r10, r1 = overall.loc["r=10"], overall.loc["r=1"]
print(f"* Louvain r=10 removes {r10['pct_removed']:.1f}% of {DATASET}'s edges but "
      f"{r10['pct_laundering_removed']:.1f}% of laundering-attempt edges (enrichment {r10['enrichment']:.2f}); "
      f"at r=1 it removes {r1['pct_removed']:.1f}% of edges and {r1['pct_laundering_removed']:.1f}% of laundering edges "
      f"(enrichment {r1['enrichment']:.2f}).")
hub = overall.loc[["hubs5", "hubs10", "hubs100"]]
print(f"* Hub removal (top 5-100) removes {hub['pct_removed'].min():.1f}-{hub['pct_removed'].max():.1f}% of edges and "
      f"{hub['pct_laundering_removed'].min():.1f}-{hub['pct_laundering_removed'].max():.1f}% of laundering edges.")

all_split = split[split["pattern_type"] == "ALL"].set_index(["dataset", "arm"])
for ds in datasets:
    d10 = all_split.loc[(ds, "r=10")]
    print(f"* {ds}, r=10: {d10['pct_split']:.1f}% of attempts are split across communities and "
          f"{d10['pct_destroyed']:.1f}% lose every two-hop path (of {int(d10['attempts_with_2path'])} with one).")

print(f"\n(AUC-ROC statistics below use the {int(keep_rows.sum())} of {len(keep_rows)} headline cells with at least "
      f"{MIN_POSITIVES} positives.)")
louvain_arms = [a for a in ORDER if a.startswith("r=") or a == "off"]
for d in DIRECTIONS:
    dl = deltas[d].loc[keep_rows, [a for a in deltas[d].columns if a in louvain_arms]]
    stacked = dl.stack()
    worst = stacked.abs().idxmax()
    print(f"* {DIR_LABEL[d]} score: largest |AUC-ROC change| vs r=10 over all arms (r=1..50 and no Louvain) is "
          f"{stacked[worst]:+.3f} ({worst[0]}, cut-off {worst[1]:g}, {worst[2]}); "
          f"median |change| {stacked.abs().median():.3f}; {(stacked.abs() < 0.02).mean() * 100:.0f}% of cells within 0.02.")
    if "off" in dl.columns:
        off = dl["off"]
        print(f"    no-Louvain arm: mean change {off.mean():+.3f}, range {off.min():+.3f} to {off.max():+.3f}.")

u, dd = cells["AUC_ROC"]["undirected"][keep_rows], cells["AUC_ROC"]["directed"][keep_rows]
wins = {a: int((u[a] > dd[a]).sum()) for a in ORDER}
lo, hi = min(wins.values()), max(wins.values())
print(f"\n* Undirected beats directed on AUC-ROC in " + (f"{lo}" if lo == hi else f"{lo}-{hi}")
      + f" of {len(u)} interpretable headline cells at every setting (see note 5 on partitions).")
if len(ties):
    g = ties.groupby("direction")["value"].agg(["min", "max"])
    print(f"* Tie group at the 100th-ranked account: {int(g['min'].min()):,}-{int(g['max'].max()):,} accounts "
          f"(of {n_test:,}) -> precision/recall/lift @K are not reported for the sweep.")

## A.6 What it costs in computation time

The pre-processing also changes how long the score takes to compute. Every measure job wrote its wall-clock time to
`timing/<dataset>_<direction>_<job>.csv`; nothing is re-run here. The timed region is the whole of stage 1: reading the
transactions and building the graph, the pre-processing (Louvain and its edge filter, or hub removal), and the block
measures of every account. No model is trained on a sweep arm, so there is no stage-2 time to add. The main figure in
A.3 carries these seconds as its bottom panel; this section tabulates them and shows what drives them.

What the scoring costs is set by each account's **second-order ego network**: for an account whose ego network holds *k*
accounts (itself included), the score builds and evaluates a dense *k* × *k* adjacency matrix. *k* is recovered exactly
from the block sizes the measure files already store, so Σ *k*² — the matrix entries built over all accounts — is a
machine-independent measure of the work, shown beside the wall-clock time.

Three limits to state with it:

* **One run per setting**, each on a shared node of the same cluster with the same number of worker processes; the node's
  load was not recorded. Read the large factors, not differences of a few tens of seconds.
* **No split by stage.** These runs predate the per-stage columns (`louvain_seconds`, `pool_seconds`, ...) that the measure
  scripts now write, so the Louvain step's own cost cannot be separated from the scoring. Hub removal runs no Louvain step
  at all, which is part of why it is the fastest. If a re-run adds the stage columns, the timing table below shows them.
* **HI-Small only**, like the rest of the sweep.

In [ ]:
# ---- Runtime per setting (loaded in A.3) beside the ego-network sizes from the block measures ----
def ego_sizes(path, direction):
    # Second-order ego-network size k of every account, from the block sizes in its measure file. The block among the
    # n1 first-order neighbours has n1(n1 - 1) free entries; the blocks between them and everything else have n1 entries
    # per other account. Undirected, the account itself sits outside those blocks (+1); directed, it is in level 0.
    # An undirected account whose one neighbour has no other neighbour has every block empty, like an isolated
    # account, and reads as k = 1 instead of 2 - immaterial to the sums below.
    if direction == "undirected":
        m = pd.read_csv(path, usecols=["size_2", "size_3"])
        within, cross, own = m["size_3"], m["size_2"], 1
    else:
        m = pd.read_csv(path, usecols=["size_01", "size_11", "size_21"])
        within, cross, own = m["size_11"], m["size_01"] + m["size_21"], 0
    n1 = np.where(within > 0, (1 + np.sqrt(1 + 4 * within)) / 2, (cross > 0).astype(float))
    k = np.where(n1 > 0, own + n1 + cross / np.where(n1 > 0, n1, 1), 1)
    return pd.Series(np.round(k))


stage_cols = [c for c in timing.columns if c.endswith("_seconds")]
display(timing.assign(order=timing["arm"].map(RT_ORDER.index)).sort_values(["direction", "order"])
        [["arm", "direction", "seconds", "n_workers", "host", "timestamp"] + stage_cols].reset_index(drop=True))
if not stage_cols:
    print("No per-stage columns in these files: the Louvain step cannot be separated from the scoring.")

ego = {}
for a in RT_ORDER:
    for d in DIRECTIONS:
        path = find(f"{arm_dataset(a)}_GARGAML_{d}.csv")
        if path is not None:
            ego[(a, d)] = ego_sizes(path, d)
work = pd.Series({key: (k ** 2).sum() for key, k in ego.items()}).unstack().reindex(index=RT_ORDER, columns=DIRECTIONS)
ego_mean = pd.Series({key: k.mean() for key, k in ego.items()}).unstack().reindex(index=RT_ORDER, columns=DIRECTIONS)
ego_max = pd.Series({key: k.max() for key, k in ego.items()}).unstack().reindex(index=RT_ORDER, columns=DIRECTIONS)
rel_rt = seconds / seconds.loc[PUBLISHED]

display(pd.concat({"seconds": seconds, "vs r=10": rel_rt.round(2), "sum k^2 (1e9)": (work / 1e9).round(2),
                   "mean k": ego_mean.round(1), "largest k": ego_max}, axis=1))
for d in DIRECTIONS:
    rho = spearmanr(seconds[d], work[d], nan_policy="omit")[0]
    print(f"{DIR_LABEL[d]}: Spearman correlation of runtime with sum k^2 over the {seconds[d].notna().sum()} settings = {rho:.2f}")

In [ ]:
# Table A9: computation time per setting.
def times(v):
    return "--" if pd.isna(v) else f"${v:.2f}\\times$"


rows = {}
for a in RT_ORDER:
    rec = [num(ego_mean.loc[a, "undirected"]), num(ego_max.loc[a, "undirected"])]
    for d in DIRECTIONS:
        rec += [num(seconds.loc[a, d]), times(rel_rt.loc[a, d])]
    rows[arm_tex(a)] = rec
tbl_rt = pd.DataFrame.from_dict(rows, orient="index", columns=[
    "\\shortstack{Ego network\\\\mean size}", "\\shortstack{Ego network\\\\largest}",
    "\\shortstack{Undirected\\\\runtime (s)}", "\\shortstack{Undirected\\\\vs $r{=}10$}",
    "\\shortstack{Directed\\\\runtime (s)}", "\\shortstack{Directed\\\\vs $r{=}10$}"])
tbl_rt.index.name = "Pre-processing"
show(tbl_rt)

# The ego-network columns are the undirected graph's; the directed graph is reduced by its own Louvain partition.
ego_gap = (ego_mean["directed"] / ego_mean["undirected"] - 1).abs().max() * 100
save_table(
    tbl_rt, f"prep_runtime_{DATASET}",
    caption=(f"Computation time of the GARG-AML block measures on {DATASET} for each pre-processing setting: wall-clock "
             "seconds for reading the data, building the graph, the pre-processing itself and the block measures of every "
             "account, and the ratio to the published setting $r=10$. The ego-network columns give the number of accounts "
             "in an account's second-order neighbourhood, itself included, which sets the size of the adjacency matrix the "
             "score evaluates for it."),
    label="tab:prep-runtime",
    note=(f"One run per setting with {n_workers} worker processes, each on a node of the same cluster; the nodes are shared "
          "and their load was not recorded, so differences of a few tens of seconds are not meaningful. Hub removal deletes "
          "the $k$ highest-degree accounts and runs no community detection, so its time contains no Louvain step. "
          "Ego-network sizes are those of the undirected reduced graph; the directed graph, reduced by its own Louvain "
          f"partition, differs by at most {ego_gap:.1f}\\% in mean size."),
    wide=True, fit=True)

In [ ]:
# Figure A6: runtime (top) and the work it tracks (bottom), per setting.
lou_rt = [a for a in RT_ORDER if not a.startswith("hubs")]
hub_rt = [a for a in RT_ORDER if a.startswith("hubs")]
xpos_rt = {a: i for i, a in enumerate(lou_rt)}
xpos_rt.update({a: len(lou_rt) + 0.7 + i for i, a in enumerate(hub_rt)})
lou_x, hub_x = [xpos_rt[a] for a in lou_rt], [xpos_rt[a] for a in hub_rt]

fig, axes = plt.subplots(2, 1, figsize=(7.0, 4.4), sharex=True)
for d in DIRECTIONS:
    style = dict(color=DIR_COLOUR[d], marker=DIR_MARKER[d], ls=DIR_STYLE[d], ms=4.5, lw=1.6)
    for ax, series in ((axes[0], seconds[d]), (axes[1], work[d])):
        ax.plot(lou_x, series.loc[lou_rt], **style)              # one line through none and the resolutions
        ax.plot(hub_x, series.loc[hub_rt], **{**style, "ls": "none"})   # hub removal is a different treatment
for ax in axes:
    ax.set_yscale("log")
    ax.axvline(len(lou_rt) - 0.15, color=INK_2, lw=0.6, ls=(0, (2, 3)), alpha=0.6)
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
axes[0].set_ylabel("stage-1 runtime (s)")
axes[1].set_ylabel("ego-matrix entries\n(sum of $k^2$)")
axes[1].set_xticks([xpos_rt[a] for a in RT_ORDER])
axes[1].set_xticklabels([arm_short(a) if a in lou_rt else a[4:] for a in RT_ORDER], fontsize=7.5)
for xs, text in ((lou_x, "Louvain r"), (hub_x, "top-k hubs")):
    axes[1].text(np.mean(xs), -0.20, text, transform=axes[1].get_xaxis_transform(), ha="center", va="top",
                 fontsize=7, color=INK_2)
handles = [Line2D([], [], color=DIR_COLOUR[d], marker=DIR_MARKER[d], ls=DIR_STYLE[d], ms=4.5, lw=1.6,
                  label=DIR_LABEL[d]) for d in DIRECTIONS]
fig.legend(handles=handles, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.04))
fig.supxlabel("Louvain resolution r (none = no reduction; † published) · number of hubs removed k, no Louvain",
              fontsize=8.5, y=-0.01)
plt.tight_layout()
save_fig(fig, "prep_runtime")
plt.show()

In [ ]:
# Numbers for the text on computation time.
print("COMPUTATION TIME - numbers for the text\n")
high_r = [a for a in lou_rt if a.startswith("r=") and float(a[2:]) > DEFAULT_RESOLUTION]
for d in DIRECTIONS:
    s, pub = seconds[d], seconds.loc[PUBLISHED, d]
    print(f"* {DIR_LABEL[d]}: {pub:,.0f} s at r=10; {s['off']:,.0f} s without Louvain ({s['off'] / pub:.1f}x), "
          f"{s['r=1']:,.0f} s at r=1 ({s['r=1'] / pub:.1f}x); {s[high_r].min():,.0f}-{s[high_r].max():,.0f} s at "
          f"r={'/'.join(a[2:] for a in high_r)}; {s[hub_rt].min():,.0f}-{s[hub_rt].max():,.0f} s with hub removal "
          f"({pub / s[hub_rt].max():.1f}-{pub / s[hub_rt].min():.1f}x faster than r=10).")
k = ego_max["undirected"]
print(f"* Largest ego network: {k['off']:,.0f} accounts without Louvain, {k['r=1']:,.0f} at r=1, {k[PUBLISHED]:,.0f} at r=10, "
      + ", ".join(f"{k[a]:,.0f} with the top {a[4:]} hubs removed" for a in hub_rt)
      + f". Removing the top 5 hubs keeps {100 - overall.loc['hubs5', 'pct_removed']:.1f}% of the edges against "
      f"{100 - overall.loc['r=1', 'pct_removed']:.1f}% at r=1, yet runs "
      f"{seconds.loc['r=1', 'undirected'] / seconds.loc['hubs5', 'undirected']:.0f}x faster (undirected): "
      "the few hubs, not the edge count, set the cost.")
if "r=50" in RT_ORDER and "hubs100" in RT_ORDER:
    print(f"* r=50 against the top-100 hubs, undirected: {seconds.loc['r=50', 'undirected']:,.0f} against "
          f"{seconds.loc['hubs100', 'undirected']:,.0f} s. Both build the same graph and score accounts with small ego "
          f"networks (largest {k['r=50']:,.0f} and {k['hubs100']:,.0f}); the gap is mostly the Louvain step, which hub "
          "removal does not run and these files do not time separately.")
print(f"\n(One run per setting, {n_workers} workers, shared nodes, no stage split - see the limits above.)")

---
# Study B — a limited view

A bank sees every transaction of its own clients, so their **first-order** neighbourhood is fully
observed — but the second-order neighbourhood the score is computed from is only visible where it runs
through those same clients. Two institutions are compared against the full network, on **their own
clients only** (labels are identical in both regimes; only the features degrade):

* `012` — the largest single bank in HI-Small (0.5 % of accounts);
* `top50` — the 50 largest banks pooled into one institution (10.8 % of accounts).

`BankObservability.ipynb` remains the notebook that builds the bank-size analysis and the coverage file;
this one digests its outputs together with `scripts/partial_observability.py`'s.

In [ ]:
accounts, metrics = {}, {}
for spec in INSTITUTIONS:
    view = view_name(DATASET, spec)
    ap = find(f"{view}_partial_observability_accounts.csv")
    mp = find(f"{view}_partial_observability_metrics.csv")
    if ap is None or mp is None:
        print(f"{spec}: files missing - run scripts/partial_observability.py; skipped")
        continue
    accounts[spec] = pd.read_csv(ap, dtype={"account": str})
    metrics[spec] = pd.read_csv(mp)
    print(f"{spec}: {accounts[spec]['account'].nunique():,} clients, read from {ap.parent.name}/ "
          f"({datetime.fromtimestamp(ap.stat().st_mtime):%Y-%m-%d})")

INST_LABEL = {"012": "Bank 012", "top50": "Top-50 banks"}

## B.1 Coverage — how much each institution sees

The laundering columns are the ceiling on what one institution can detect at all: a laundering
transaction outside its view is out of reach of any score.

In [ ]:
cov_path = find("appendix_coverage.csv")
if cov_path is None:
    print("appendix_coverage.csv missing - run notebooks/BankObservability.ipynb; skipping B.1")
else:
    cov = pd.read_csv(cov_path, dtype={"institution": str}).set_index("institution")
    rows = {}
    for spec, r in cov.iterrows():
        rows[INST_LABEL.get(spec, spec)] = [
            num(r["banks"]), f"{num(r['clients'])} ({r['pct_clients']:.1f}\\%)",
            f"{num(r['transactions'])} ({r['pct_transactions']:.1f}\\%)",
            f"{num(r['laundering'])} ({r['pct_laundering']:.1f}\\%)", num(r["external"])]
    tbl_cov = pd.DataFrame.from_dict(rows, orient="index", columns=[
        "Banks", "Clients (\\% of accounts)", "Transactions (\\% of all)",
        "Laundering transactions (\\% of all)", "External counterparties"])
    tbl_cov.index.name = "Institution"
    show(tbl_cov)
    save_table(
        tbl_cov, f"obs_coverage_{DATASET}",
        caption=(f"What each institution observes on {DATASET}. An institution sees every transaction of its own "
                 "clients; external counterparties enter its graph as neighbours but are never scored."),
        label="tab:obs-coverage",
        note="The share of laundering transactions visible bounds the recall any score computed at that institution can reach.",
        wide=True, fit=True)

## B.2 Neighbourhood survival — the mechanism

The control and the effect in one table. **N₁ survival is the control**: a bank sees every transaction of its
own clients, so their direct counterparties are fully observable and N₁ survival must be 1. The only
legitimate exception is an account held at two banks, one of them outside the institution; the table counts the
clients where it is below 1, and a count above zero deserves a look rather than a rounding. **N₂ survival is what
partial observability destroys**, and N₂ is what the block densities are computed from.

N₂ survival is the per-client ratio |N₂ in view| / |N₂ full|; clients with no second-order neighbourhood on the
full graph have no ratio and are left out of its mean and median. "Lose all N₂" is a share **of the clients that have
one**; the column before it gives the share that have none even on the full network, so that the two together
reconcile with a share taken over every client.

In [ ]:
und = {spec: df[df["direction"] == "undirected"].copy() for spec, df in accounts.items()}

rows = {}
survival = {}
for spec, df in und.items():
    n1 = df["n1_view"] / df["n1_full"].replace(0, np.nan)
    has_n2 = df["n2_full"] > 0
    n2 = (df["n2_view"] / df["n2_full"].replace(0, np.nan))
    survival[spec] = n2.dropna()
    rows[INST_LABEL[spec]] = [
        num(len(df)), num((n1 < 1).sum()), num(df["n2_full"].median()), num(df["n2_view"].median()),
        fmt(n2.mean()), fmt(n2.median(), 4),
        f"{100 * (~has_n2).mean():.1f}\\%",
        f"{100 * (df.loc[has_n2, 'n2_view'] == 0).mean():.1f}\\%",
        f"{100 * (df.loc[has_n2, 'n2_view'] == df.loc[has_n2, 'n2_full']).mean():.1f}\\%"]
tbl_surv = pd.DataFrame.from_dict(rows, orient="index", columns=[
    "Clients", "Clients with $N_1$ survival $<1$", "Median $|N_2|$ full", "Median $|N_2|$ view",
    "Mean $N_2$ survival", "Median $N_2$ survival", "No $N_2$ on full network", "Lose all $N_2$",
    "Keep all $N_2$"])
tbl_surv.index.name = "Institution"
show(tbl_surv)
save_table(
    tbl_surv, f"obs_survival_{DATASET}",
    caption=(f"Survival of the second-order neighbourhood in an institution's view of {DATASET}. $N_1$ survival is a "
             "control: every client's direct counterparties are observed, except where the account is held at a bank "
             "outside the institution. The last two columns are shares of the clients that have a second-order "
             "neighbourhood on the full network."),
    label="tab:obs-survival", wide=True, fit=True)

In [ ]:
# Figure B1: distribution of N2 survival.
fig, axes = plt.subplots(1, len(survival), figsize=(3.5 * len(survival), 2.9), sharey=True, squeeze=False)
for ax, (spec, s) in zip(axes[0], survival.items()):
    ax.hist(s, bins=np.linspace(0, 1, 26), weights=np.full(len(s), 1 / len(s)), color=INK_2,
            edgecolor="white", linewidth=0.6)
    ax.axvline(s.mean(), color=INK, ls="--", lw=1)
    ax.text(0.97, 0.95, f"mean {s.mean():.2f}\nmedian {s.median():.3f}", transform=ax.transAxes, ha="right",
            va="top", fontsize=8)
    ax.set_xlabel("share of $N_2$ still visible")
    ax.set_title(INST_LABEL[spec])
axes[0, 0].set_ylabel("share of clients")
plt.tight_layout()
save_fig(fig, "obs_n2_survival")
plt.show()

## B.3 What happens to the score

How far the score moves when the institution loses its second-order view. Two things matter and they
are different: the **ranking** (Spearman ρ — does the same client stay on top?) and the **level**
(mean shift — does the score drift?). Losing N₂ can empty the penalty blocks and push the score
*upward*, which would flag more clients rather than just rank them differently; how much it does depends on how
much of N₂ survives (B.2), so read the shift beside the survival table. The `degree ≥ 3` subgroup removes clients that
are structurally degenerate (too few neighbours to have a second-order neighbourhood worth scoring).

In [ ]:
MIN_DEGREE = 3
SUBGROUPS = [("all", None), (f"degree>={MIN_DEGREE}", MIN_DEGREE)]
rows = {}
for spec, df in accounts.items():
    for d in DIRECTIONS:
        dd_ = df[df["direction"] == d]
        for name, k in SUBGROUPS:
            s = dd_ if k is None else dd_[dd_["degree"] >= k]
            f, v = s["GARGAML_full"], s["GARGAML_view"]
            ok = f.notna() & v.notna()
            f, v = f[ok], v[ok]
            rows[(INST_LABEL[spec], DIR_LABEL[d], "all" if k is None else f"$\\geq {k}$")] = [
                num(len(f)), fmt(f.mean()), fmt(v.mean()), fmt((v - f).mean()),
                fmt(spearmanr(f, v)[0], 2), f"{100 * np.isclose(f, v).mean():.1f}\\%"]
tbl_shift = pd.DataFrame.from_dict(rows, orient="index", columns=[
    "Clients", "Mean full", "Mean view", "Mean shift", "Spearman $\\rho$", "Unchanged"])
tbl_shift.index = pd.MultiIndex.from_tuples(tbl_shift.index, names=["Institution", "Score", "Degree"])
show(tbl_shift)
save_table(
    tbl_shift, f"obs_score_shift_{DATASET}",
    caption=("Effect of the limited view on the GARG-AML score of an institution's own clients. Spearman $\\rho$ "
             "compares the ranking of clients between the full network and the institution's view; "
             "\\emph{Unchanged} is the share of clients whose score is identical in both."),
    label="tab:obs-score-shift", wide=True)

In [ ]:
def plot_shift(direction, stem):
    specs = list(accounts)
    fig, axes = plt.subplots(2, len(specs), figsize=(3.6 * len(specs), 6.0), squeeze=False)
    for col, spec in enumerate(specs):
        df = accounts[spec]
        df = df[(df["direction"] == direction) & (df["degree"] >= MIN_DEGREE)]
        f, v = df["GARGAML_full"], df["GARGAML_view"]
        ax = axes[0, col]
        hb = ax.hexbin(f, v, gridsize=32, extent=(-1, 1, -1, 1), bins="log", cmap=SEQ, mincnt=1, linewidths=0.2)
        ax.plot([-1, 1], [-1, 1], ls="--", lw=1, color=INK)
        ax.set_xlim(-1.05, 1.05)
        ax.set_ylim(-1.05, 1.05)
        ax.set_xlabel("score, full network")
        ax.set_ylabel("score, institution's view")
        ax.set_title(f"{INST_LABEL[spec]}  (degree $\\geq$ {MIN_DEGREE}, $\\rho$ = {spearmanr(f, v)[0]:.2f})")
        cb = fig.colorbar(hb, ax=ax, fraction=0.045, pad=0.02)
        cb.set_label("clients (log)", fontsize=7.5)
        cb.outline.set_visible(False)
        ax = axes[1, col]
        bins = np.linspace(-1, 1, 41)
        ax.hist(f, bins=bins, color=NEUTRAL, alpha=0.9, label="full network", edgecolor="white", linewidth=0.4)
        ax.hist(v, bins=bins, histtype="step", color=DIR_COLOUR[direction], linewidth=1.6, label="institution's view")
        ax.set_xlabel("GARG-AML score")
        ax.set_ylabel("clients")
        ax.set_ylim(top=ax.get_ylim()[1] * 1.3)
        ax.legend(loc="upper left")
    plt.tight_layout()
    save_fig(fig, stem)
    plt.show()


if accounts:
    plot_shift("undirected", "obs_score_shift_undirected")
    plot_shift("directed", "obs_score_shift_directed")

## B.4 Detection

Ranking quality of the score, full network vs the institution's view, for the institution's own clients.
The label cut-offs are those with enough positives on one institution's clients (0.0 and 0.1);
pattern-specific targets are reported as gaps by the pipeline where positives are too few.

**AUC-PR relative to prevalence** puts every cell on the same footing: 1 is a random ranking.

**Read the number of positives before the AUC.** One institution's clients carry few laundering accounts —
for the single bank at cut-off 0.1 the *Positives* column shows how few — so a large AUC change in such a cell is
within what the sampling of a handful of positives can produce. There are no confidence intervals here; the tables therefore
carry the count, and a cell resting on a few dozen positives should not be quoted as an effect. Several full-network
AUC-ROCs also sit **below 0.5**: the comparison is then between two weak rankings, not between a good one and a
degraded one.

In [ ]:
def lookup(m):
    ok = m[m["status"] == "ok"].copy()
    ok["K"] = ok["K"].fillna(-1)
    return ok.set_index(["direction", "regime", "subgroup", "cutoff", "target", "metric", "K"])["value"].to_dict()


LOOK = {spec: lookup(m) for spec, m in metrics.items()}


def value(spec, direction, regime, subgroup, cutoff, target, metric, K=-1):
    return LOOK[spec].get((direction, regime, subgroup, cutoff, target, metric, float(K)), np.nan)


def positives(spec, subgroup, cutoff, target):
    m = metrics[spec]
    r = m[(m["subgroup"] == subgroup) & (m["cutoff"] == cutoff) & (m["target"] == target)
          & (m["metric"] == "AUC_ROC")]
    return (int(r["n_pos"].iloc[0]), int(r["n_test"].iloc[0])) if len(r) else (np.nan, np.nan)


def prevalence(spec, subgroup, cutoff, target):
    n_pos, n_test = positives(spec, subgroup, cutoff, target)
    return n_pos / n_test


CUTOFFS_B = sorted({c for m in metrics.values() for c in m["cutoff"].unique()})
status = pd.concat([m.assign(institution=s) for s, m in metrics.items()])
print("rows by status:", status["status"].value_counts().to_dict())
print("targets with at least one 'ok' AUC-ROC:",
      sorted(status[(status["metric"] == "AUC_ROC") & (status["status"] == "ok")]["target"].unique()))

In [ ]:
def detection_table(subgroup="all", target="Is Laundering"):
    rows = {}
    for spec in accounts:
        for d in DIRECTIONS:
            for c in CUTOFFS_B:
                prev = prevalence(spec, subgroup, c, target)
                thin = positives(spec, subgroup, c, target)[0] < MIN_POSITIVES
                rows[(INST_LABEL[spec], DIR_LABEL[d], f"{c:g}")] = [
                    fmt(value(spec, d, "full", subgroup, c, target, "AUC_ROC"), thin=thin),
                    fmt(value(spec, d, "view", subgroup, c, target, "AUC_ROC"), thin=thin),
                    fmt(value(spec, d, "full", subgroup, c, target, "AUC_PR"), thin=thin),
                    fmt(value(spec, d, "view", subgroup, c, target, "AUC_PR"), thin=thin), fmt(prev),
                    num(positives(spec, subgroup, c, target)[0])]
    t = pd.DataFrame.from_dict(rows, orient="index", columns=[
        "AUC-ROC full", "AUC-ROC view", "AUC-PR full", "AUC-PR view", "Prevalence", "Positives"])
    t.index = pd.MultiIndex.from_tuples(t.index, names=["Institution", "Score", "Cut-off"])
    return t


for subgroup, tag, text in [("all", "all", "all clients"),
                            (f"degree>={MIN_DEGREE}", f"degree{MIN_DEGREE}", f"clients with degree $\\geq {MIN_DEGREE}$")]:
    t = detection_table(subgroup)
    print(f"\n--- {subgroup} ---")
    show(t)
    save_table(
        t, f"obs_detection_{tag}_{DATASET}",
        caption=(f"AUC-ROC and AUC-PR of the GARG-AML score for an institution's own clients ({text}), on the full "
                 "network and on the institution's view. Prevalence is the share of positives at that label cut-off; "
                 "an AUC-PR at the prevalence is a random ranking. Positives is the number of positive clients the metrics rest on."),
        label=f"tab:obs-detection-{tag}",
        note=(f"$^{{*}}$ Fewer than {MIN_POSITIVES} positives: shown, not interpretable. Louvain is off on both sides "
              "of the comparison, so the full-network column is not the published result."),
        wide=True, fit=True)

In [ ]:
# Figure B2: full -> view, per cut-off, institution and score (dumbbells).
combos = [(spec, c) for spec in accounts for c in CUTOFFS_B]
ypos = {combo: len(combos) - 1 - i for i, combo in enumerate(combos)}
offset = {"undirected": 0.14, "directed": -0.14}

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3), sharey=True)
for ax, metric, title, ref in [(axes[0], "AUC_ROC", "AUC-ROC", 0.5),
                               (axes[1], "AUC_PR", "AUC-PR / prevalence", 1.0)]:
    for (spec, c), y in ypos.items():
        for d in DIRECTIONS:
            a = value(spec, d, "full", "all", c, "Is Laundering", metric)
            b = value(spec, d, "view", "all", c, "Is Laundering", metric)
            if metric == "AUC_PR":
                prev = prevalence(spec, "all", c, "Is Laundering")
                a, b = a / prev, b / prev
            yy = y + offset[d]
            al = 0.4 if positives(spec, "all", c, "Is Laundering")[0] < MIN_POSITIVES else 1.0
            ax.plot([a, b], [yy, yy], color=DIR_COLOUR[d], lw=1.6, zorder=2, alpha=al)
            ax.scatter([a], [yy], s=26, facecolor="white", edgecolor=DIR_COLOUR[d], linewidth=1.4,
                       marker=DIR_MARKER[d], zorder=3, alpha=al)
            ax.scatter([b], [yy], s=26, color=DIR_COLOUR[d], marker=DIR_MARKER[d], zorder=3, alpha=al)
    ax.axvline(ref, color=INK_2, lw=0.8, ls=":")
    ax.set_xlabel(title)
    ax.grid(axis="y", visible=False)
axes[0].set_yticks([ypos[k] for k in combos])
axes[0].set_yticklabels([f"{INST_LABEL[s]} · cut-off {c:g}\n({positives(s, 'all', c, 'Is Laundering')[0]} positives)"
                         for s, c in combos], fontsize=7.5)
handles = ([Line2D([], [], color=DIR_COLOUR[d], marker=DIR_MARKER[d], lw=1.6, ms=5, label=DIR_LABEL[d])
            for d in DIRECTIONS]
           + [Line2D([], [], color=INK_2, marker="o", mfc="white", ls="", ms=5, label="full network"),
              Line2D([], [], color=INK_2, marker="o", ls="", ms=5, label="institution's view")])
fig.legend(handles=handles, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.08))
if any(positives(s, "all", c, "Is Laundering")[0] < MIN_POSITIVES for s, c in combos):
    fig.text(0.5, -0.03, f"faded: fewer than {MIN_POSITIVES} positives, not interpretable", ha="center", fontsize=8)
plt.tight_layout()
save_fig(fig, "obs_detection")
plt.show()

## B.5 Alert volume and the alert queue

Two views of what an investigator would actually see.

* **Fixed rule.** Flag every client with score `> 0.5` — the same rule as the base-score precision / F1
  tables. Because the limited view biases the score upward, this is where the cost shows up as *more
  alerts*, not merely a different order. The positive class is `propensity > 0` (cut-off 0.0).
* **Top-K queue.** Precision at K=100 beside `ties@K`. Wherever the tie group at the cut-off exceeds K,
  "the top K" is not uniquely defined and the number is not interpretable on its own.

In [ ]:
def flags(df, regime, cutoff=0.0):
    s = df[f"GARGAML_{regime}"]
    ok = s.notna()
    y = df.loc[ok, "Is Laundering"] > cutoff
    hit = s[ok] > SCORE_THRESHOLD
    tp, n_flag, n_pos = int((hit & y).sum()), int(hit.sum()), int(y.sum())
    prec = tp / n_flag if n_flag else np.nan
    rec = tp / n_pos if n_pos else np.nan
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else np.nan
    modal = s[ok].round(9).value_counts().iloc[0] / ok.sum()
    return dict(flagged=n_flag, tp=tp, precision=prec, recall=rec, f1=f1, modal=modal, n_pos=n_pos, n=int(ok.sum()))


rows, plain = {}, []
for spec, df in accounts.items():
    for d in DIRECTIONS:
        sub = df[df["direction"] == d]
        full, view = flags(sub, "full"), flags(sub, "view")
        plain.append(dict(institution=spec, direction=d, **{f"{k}_full": v for k, v in full.items()},
                          **{f"{k}_view": v for k, v in view.items()}))
        rows[(INST_LABEL[spec], DIR_LABEL[d])] = [
            num(full["flagged"]), num(view["flagged"]), fmt(full["precision"]), fmt(view["precision"]),
            fmt(full["recall"]), fmt(view["recall"]),
            f"{100 * full['modal']:.1f}\\%", f"{100 * view['modal']:.1f}\\%"]
tbl_flag = pd.DataFrame.from_dict(rows, orient="index", columns=[
    "Flagged full", "Flagged view", "Precision full", "Precision view", "Recall full", "Recall view",
    "Modal score full", "Modal score view"])
tbl_flag.index = pd.MultiIndex.from_tuples(tbl_flag.index, names=["Institution", "Score"])
flag_numbers = pd.DataFrame(plain)
show(tbl_flag)
save_table(
    tbl_flag, f"obs_alerts_{DATASET}",
    caption=(f"Clients flagged by the rule score $> {SCORE_THRESHOLD}$ on the full network and on the institution's view, "
             "with the precision and recall of that rule against `propensity $> 0$'. The last two columns give the share of "
             "clients sharing the most common score value, i.e. how tied the ranking is."),
    label="tab:obs-alerts",
    note="Louvain is off on both sides of the comparison.", wide=True, fit=True)

In [ ]:
K_QUEUE = 100
rows = {}
for spec in accounts:
    for d in DIRECTIONS:
        row = []
        for regime in ("full", "view"):
            row.append(fmt(value(spec, d, regime, "all", 0.0, "Is Laundering", "P@K", K_QUEUE)))
        for regime in ("full", "view"):
            ties = value(spec, d, regime, "all", 0.0, "Is Laundering", "ties@K", K_QUEUE)
            row.append("--" if pd.isna(ties) else num(ties) + ("$^{\\ddagger}$" if ties > K_QUEUE else ""))
        rows[(INST_LABEL[spec], DIR_LABEL[d])] = row
tbl_queue = pd.DataFrame.from_dict(rows, orient="index", columns=[
    f"P@{K_QUEUE} full", f"P@{K_QUEUE} view", f"ties@{K_QUEUE} full", f"ties@{K_QUEUE} view"])
tbl_queue.index = pd.MultiIndex.from_tuples(tbl_queue.index, names=["Institution", "Score"])
show(tbl_queue)
save_table(
    tbl_queue, f"obs_queue_{DATASET}",
    caption=(f"Precision at K={K_QUEUE} for an institution's own clients (positive class: propensity $> 0$), beside the "
             "number of clients sharing the score of the K-th ranked client."),
    label="tab:obs-queue",
    note=("$^{\\ddagger}$ The tie group at the cut-off exceeds $K$: the top-$K$ is not uniquely defined and the "
          "precision depends on how ties are broken."))

In [ ]:
# ---- Study B: numbers for the text -------------------------------------------------
print("PARTIAL OBSERVABILITY - numbers for the text\n")
for spec, df in und.items():
    n1 = df["n1_view"] / df["n1_full"].replace(0, np.nan)
    s = survival[spec]
    has = df["n2_full"] > 0
    print(f"* {INST_LABEL[spec]}: {len(df):,} clients; {int((n1 < 1).sum())} with N1 survival below 1 (control: expect 0, "
          f"or accounts held at a bank outside the institution); N2 survival mean {s.mean():.3f}, median {s.median():.4f}; "
          f"{100 * (df.loc[has, 'n2_view'] == 0).mean():.0f}% of clients that have an N2 lose all of it "
          f"({100 * (df['n2_view'] == 0).mean():.0f}% of all clients end with none).")
print()
for _, r in flag_numbers.iterrows():
    print(f"* {INST_LABEL[r['institution']]}, {r['direction']}: flagged at score>{SCORE_THRESHOLD} "
          f"{r['flagged_full']:,} -> {r['flagged_view']:,} "
          f"({r['flagged_view'] / max(r['flagged_full'], 1):.2f}x); precision "
          f"{r['precision_full']:.3f} -> {r['precision_view']:.3f}; recall {r['recall_full']:.3f} -> {r['recall_view']:.3f}.")
print()
for spec in accounts:
    for d in DIRECTIONS:
        for c in CUTOFFS_B:
            a = value(spec, d, "full", "all", c, "Is Laundering", "AUC_ROC")
            b = value(spec, d, "view", "all", c, "Is Laundering", "AUC_ROC")
            print(f"* {INST_LABEL[spec]}, {d}, cut-off {c:g}: AUC-ROC {a:.3f} -> {b:.3f} ({b - a:+.3f})")

---
# Output manifest

Everything written by this notebook, ready to copy into the manuscript project. Each `.tex` table
carries its own caption and label (`tab:*`); figures are vector PDFs with embedded fonts.

In [ ]:
manifest = pd.DataFrame(
    [dict(file=p.name, kind="figure" if p.suffix == ".pdf" else "table (tex)" if p.suffix == ".tex" else "table (csv)",
          kb=round(p.stat().st_size / 1024, 1)) for p in sorted(set(WRITTEN)) if p.exists()])
display(manifest)
print(f"{len(manifest)} files in {OUT}")

---
# To include in paper

The selection, made against what the reviewers asked:

* **Main body — the pre-processing ablation** (Louvain resolution, no Louvain, hub removal). Reviewer R2-M3 asked for a
  sensitivity analysis over the resolution *and* the edge-removal step, noting that a pattern straddling two communities
  is destroyed; the JMLC comment asked for the share of edges severed. It is a small section, so it gets **one table and
  one figure**.
* **Appendix — the limited view.** Reviewers R1-1 and R1-2 asked for an experiment simulating partial observability. It
  is an appendix, so it is kept to **three items**: who the institutions are, what the view destroys, and what that does
  to detection.

Everything else stays in `paper_robustness/` as supplementary material and is listed at the end of each part with the
reason it was held back. The selected files are also copied into `paper_robustness/selected/` so that one folder can be
uploaded to the manuscript project.

In [ ]:
import shutil

from IPython.display import Image, display

pd.set_option("display.max_colwidth", 170)
SELECTED = OUT / "selected"
SELECTED.mkdir(exist_ok=True)

MAIN, APPX = "Main body: pre-processing ablation", "Appendix: limited view"
SELECTION = [
    dict(part=MAIN, kind="Table", file=f"table_prep_ablation_{DATASET}.tex", label="tab:prep-ablation", answers="R2-M3",
         why="Cost and effect of every setting in one place: edges removed, laundering edges removed, attempts that lose "
             "their two-hop path (the 'pattern straddling two communities' concern) and the score's AUC-ROC."),
    dict(part=MAIN, kind="Figure", file="fig_sweep_score_any-laundering.pdf", label="fig:sweep-score", answers="R2-M3",
         why="The sensitivity itself: AUC-ROC and AUC-PR over the whole sweep, both directions, four cut-offs, and "
             "under them what each setting costs in stage-1 runtime. Every score panel rests on at least "
             f"{int(n_pos.loc['Is Laundering'].min())} positives; the runtime is one run per setting (see A.6)."),
    dict(part=APPX, kind="Table", file=f"table_obs_coverage_{DATASET}.tex", label="tab:obs-coverage", answers="R1-1, R1-2",
         why="Defines the two institutions and the ceiling on what they can detect; two rows, so it costs almost no space."),
    dict(part=APPX, kind="Figure", file="fig_obs_n2_survival.pdf", label="fig:obs-n2-survival", answers="R1-1, R1-2",
         why="The mechanism: how much of the second-order neighbourhood, which the score is computed from, survives."),
    dict(part=APPX, kind="Figure", file="fig_obs_detection.pdf", label="fig:obs-detection", answers="R1-1, R1-2",
         why="The effect: full network against institution's view, with the number of positives on every row and "
             "thin rows faded."),
]
selection = pd.DataFrame(SELECTION)
missing_files = [f for f in selection["file"] if not (OUT / f).exists()]
assert not missing_files, f"selected files not written: {missing_files}"
for f in selection["file"]:
    shutil.copy2(OUT / f, SELECTED / f)
print(f"copied {len(selection)} files to {SELECTED}\n")
display(selection)


def figure_snippet(file, label, caption, star=True):
    env = "figure*" if star else "figure"
    return "\n".join([f"\\begin{{{env}}}[t]", "\\centering",
                      f"\\includegraphics[width=\\linewidth]{{figures/{file[:-4]}}}",
                      f"\\caption{{{caption}}}", f"\\label{{{label}}}", f"\\end{{{env}}}"])


def table_snippet(file):
    return f"\\input{{tables/{file[:-4]}}}"

## Main body — the pre-processing ablation

**Table.** One row per setting; the cost columns and the effect columns side by side, so a reader sees at a glance that the
choice of resolution moves the score little while the removal of edges is large. Hub removal is the second arm of the
ablation, so it sits in the same table rather than in a section of its own.

In [ ]:
show(TABLES[f"prep_ablation_{DATASET}"])
print("LaTeX (paths are relative to wherever you put the selected/ folder):\n")
print(table_snippet(f"table_prep_ablation_{DATASET}.tex"))

**Figure.** Both directions on one set of axes in each panel, so the comparison the reviewers care about — does the score
survive a different pre-processing? — is a visual one. The cut-offs are columns because their positives differ by two
orders of magnitude. The runtime panel underneath spans the cut-offs, since a setting's computation time does not
depend on the label: it puts what each setting costs directly under what it does to the score.

In [ ]:
display(Image(filename=str(OUT / "preview" / "fig_sweep_score_any-laundering.png"), width=780))

caption_sweep = (
    "AUC-ROC (top) and AUC-PR relative to the prevalence of positives (middle) of the GARG-AML score on HI-Small, for each "
    "pre-processing setting and four label cut-offs, and the wall-clock time of stage 1 for each setting (bottom, in seconds: "
    f"building the graph, the pre-processing and the block measures of every account, one run each with {n_workers} worker "
    "processes; it does not depend on the cut-off). Undirected in blue, directed in orange; \"none\" is no Louvain "
    "reduction, the numbers are Louvain resolutions and \u2020 marks the published setting ($r=10$); right of the dashed "
    "line, the top-$k$ hubs are removed instead of running Louvain. No model is fitted: every score is a full-population "
    "value. Dotted lines mark a random ranking.")
print("LaTeX:\n")
print(figure_snippet("fig_sweep_score_any-laundering.pdf", "fig:sweep-score", caption_sweep))

In [ ]:
# Numbers for the text of the main-body section, computed from the notebook's own results.
print("MAIN BODY - numbers for the text\n")
r10, r1 = overall.loc["r=10"], overall.loc["r=1"]
hub = overall.loc[["hubs5", "hubs10", "hubs100"]]
lost_r10 = split[(split["pattern_type"] == "ALL") & (split["arm"] == "r=10")].set_index("dataset")["pct_destroyed"]
r_only = [a for a in ORDER if a.startswith("r=")]
mean_auc = {d: cells["AUC_ROC"][d][keep_rows].mean() for d in DIRECTIONS}

print("Cost")
print(f"* Louvain r=10 removes {r10['pct_removed']:.1f}% of the edges and {r10['pct_laundering_removed']:.1f}% of the "
      f"laundering-attempt edges (enrichment {r10['enrichment']:.2f}); r=1 removes {r1['pct_removed']:.1f}% and "
      f"{r1['pct_laundering_removed']:.1f}% (enrichment {r1['enrichment']:.2f}).")
print(f"* Hub removal (top 5-100) removes {hub['pct_removed'].min():.1f}-{hub['pct_removed'].max():.1f}% of the edges and "
      f"{hub['pct_laundering_removed'].min():.1f}-{hub['pct_laundering_removed'].max():.1f}% of the laundering edges.")
print("* Laundering attempts that lose every two-hop path at r=10: "
      + ", ".join(f"{ds} {v:.1f}%" for ds, v in lost_r10.items())
      + ". The LI-Large figure is the limitation to state; the sweep itself runs on HI-Small only.")

print("\nEffect on the score (mean AUC-ROC over the interpretable cells)")
for d in DIRECTIONS:
    m = mean_auc[d]
    dl = deltas[d].loc[keep_rows]
    sweep = dl[[a for a in dl.columns if a in r_only]].stack()
    print(f"* {DIR_LABEL[d]}: {m['off']:.3f} without Louvain, {m[PUBLISHED]:.3f} at r=10, "
          f"{m[r_only].min():.3f}-{m[r_only].max():.3f} across r=1..50; largest single-cell change vs r=10 is "
          f"{sweep.abs().max():.3f} across the resolutions and {dl['off'].abs().max():.3f} without Louvain.")
u, dd = cells["AUC_ROC"]["undirected"][keep_rows], cells["AUC_ROC"]["directed"][keep_rows]
print(f"* Undirected exceeds directed in {min(int((u[a] > dd[a]).sum()) for a in ORDER)}"
      f" of {len(u)} interpretable cells at every setting (a gap between directions at one resolution is not purely a "
      "direction effect: the two can be reduced by different Louvain partitions).")

print("\nHub removal")
hub_arms = [a for a in ORDER if a.startswith("hubs")]
if hub_arms:
    for d in DIRECTIONS:
        m = mean_auc[d]
        print(f"* {DIR_LABEL[d]}: mean AUC-ROC " + ", ".join(f"{m[a]:.3f} (top {a[4:]})" for a in hub_arms)
              + f" against {m[PUBLISHED]:.3f} at r=10 and {m['off']:.3f} without Louvain.")
    print("* Attempts losing every two-hop path under hub removal: "
          + ", ".join(f"top {a[4:]} {lost.get(a, np.nan):.1f}%" for a in hub_arms)
          + f" (Louvain r=10: {lost['r=10']:.1f}%).")
    print("* The deleted hubs are laundering-heavy: "
          + "; ".join(f"top {a[4:]} removes {int(lost_pos[a])} accounts with a laundering transaction" for a in hub_arms)
          + ". Hub removal destroys no two-hop attempt; the per-pattern heatmap in the supplement shows the laundering edges it "
          "removes are overwhelmingly unclassified ones. The hub rows are scored on the accounts that remain.")
else:
    print("* NOT YET AVAILABLE - the hub arms have edge counts but no scored metrics, so their AUC-ROC cells in the table "
          "are dashes. Score the arms, or drop the three hub rows and keep hub removal to a sentence on edges removed.")

**Held back for the supplement** (everything is already written to `paper_robustness/`):

In [ ]:
HELD_MAIN = [
    ("fig_prep_pattern_splitting.pdf", "Splitting by pattern type on both datasets. The one worth the space if the main "
     "text quotes the LI-Large figure: it is the direct answer to 'a pattern straddling two communities is destroyed'."),
    ("table_prep_severance_HI-Small.tex, fig_prep_severance.pdf", "The first two columns of the ablation table, plus the enrichment."),
    ("fig_prep_severance_patterns.pdf, table_prep_severance_patterns_HI-Small.tex", "Per-pattern detail behind the 'laundering "
     "edges removed' column; hub removal spares the identified patterns and removes unclassified laundering edges instead."),
    ("fig_sweep_score_gather-scatter.pdf, fig_sweep_score_scatter-gather.pdf", "Pattern-specific sweeps; their high "
     "cut-offs rest on a handful of positives (see the positives table in A.3)."),
    ("fig_sweep_score_delta_auc_roc.pdf", "Cell-level deviations from r=10; the ablation table's mean plus one sentence say "
     "the same more briefly."),
    ("table_sweep_score_summary_HI-Small.tex, table_sweep_score_HI-Small_AUC_ROC.tex, ..._AUC_PR.tex",
     "The ablation table's AUC columns in more detail (max deviation; the full grid)."),
    ("table_sweep_score_ties_HI-Small.tex", "Justifies not reporting P@K for the sweep; one sentence in the text is enough."),
    ("table_prep_runtime_HI-Small.tex, fig_prep_runtime.pdf", "The runtime panel of the main figure as a table, with the "
     "ego-network sizes behind it; the figure adds the work measure (sum of k^2) that explains the runtime."),
]
display(pd.DataFrame(HELD_MAIN, columns=["file", "why it was held back"]))

---
## Appendix — the limited view

Kept to a table that defines the institutions, a figure for the mechanism and a figure for the effect. The text can carry
the rest in two or three sentences, taken from the numbers below.

In [ ]:
show(TABLES[f"obs_coverage_{DATASET}"])
print("LaTeX:\n")
print(table_snippet(f"table_obs_coverage_{DATASET}.tex"))

In [ ]:
display(Image(filename=str(OUT / "preview" / "fig_obs_n2_survival.png"), width=620))

caption_n2 = (
    "Share of each client's second-order neighbourhood that is still visible in the institution's view of HI-Small: bank 012 "
    "(left) and the 50 largest banks pooled (right). The dashed line is the mean. Clients with no second-order "
    "neighbourhood on the full network are not shown.")
print("LaTeX:\n")
print(figure_snippet("fig_obs_n2_survival.pdf", "fig:obs-n2-survival", caption_n2, star=False))

In [ ]:
display(Image(filename=str(OUT / "preview" / "fig_obs_detection.png"), width=780))

caption_det = (
    "AUC-ROC (left) and AUC-PR relative to prevalence (right) of the GARG-AML score for an institution's own clients, on the "
    "full network (open markers) and on the institution's view (filled markers), undirected and directed, at two label "
    "cut-offs. Louvain is off on both sides, so the full-network values are not the published results. Faded rows rest on "
    "fewer than 30 positives and are not interpretable; dotted lines mark a random ranking.")
print("LaTeX:\n")
print(figure_snippet("fig_obs_detection.pdf", "fig:obs-detection", caption_det))

In [ ]:
# Numbers for the text of the appendix.
print("APPENDIX - numbers for the text\n")
if cov_path is not None:
    for spec, r in cov.iterrows():
        print(f"* {INST_LABEL.get(spec, spec)}: {r['clients']:,.0f} clients = {r['pct_clients']:.1f}% of accounts; sees "
              f"{r['pct_transactions']:.1f}% of transactions and {r['pct_laundering']:.1f}% of laundering transactions "
              f"({r['laundering']:,.0f}), which caps the recall any score computed there can reach.")
print()
for spec, df in und.items():
    s_ = survival[spec]
    has = df["n2_full"] > 0
    print(f"* {INST_LABEL[spec]}: mean N2 survival {s_.mean():.2f} (median {s_.median():.3f}); "
          f"{100 * (df.loc[has, 'n2_view'] == 0).mean():.0f}% of the clients that have a second-order neighbourhood lose all of it.")
print()
thin_rows, full_below_half = [], 0
for spec in accounts:
    for d in DIRECTIONS:
        for c in CUTOFFS_B:
            n = positives(spec, "all", c, "Is Laundering")[0]
            a = value(spec, d, "full", "all", c, "Is Laundering", "AUC_ROC")
            b = value(spec, d, "view", "all", c, "Is Laundering", "AUC_ROC")
            full_below_half += a < 0.5
            if n < MIN_POSITIVES:
                thin_rows.append(f"{INST_LABEL[spec]}, cut-off {c:g} ({n} positives)")
                continue
            print(f"* {INST_LABEL[spec]}, {d}, cut-off {c:g} ({n:,} positives): AUC-ROC {a:.3f} -> {b:.3f} ({b - a:+.3f})")
print(f"\nNot interpreted (fewer than {MIN_POSITIVES} positives): " + "; ".join(sorted(set(thin_rows))) + ".")
print(f"{full_below_half} of {len(accounts) * len(DIRECTIONS) * len(CUTOFFS_B)} full-network AUC-ROCs are below 0.5: "
      "the score is already a weak ranking on one institution's clients, so the comparison is between two weak rankings.")
for _, r in flag_numbers[flag_numbers["direction"] == "undirected"].iterrows():
    print(f"* Flagged at score > {SCORE_THRESHOLD}, undirected, {INST_LABEL[r['institution']]}: "
          f"{r['flagged_full']:,} -> {r['flagged_view']:,} ({r['flagged_view'] / max(r['flagged_full'], 1):.2f}x).")

**Held back for the supplement:**

In [ ]:
HELD_APPX = [
    ("table_obs_survival_HI-Small.tex", "The numbers behind the survival figure, with the N1 control; quote it in the text instead."),
    ("fig_obs_score_shift_undirected.pdf, fig_obs_score_shift_directed.pdf, table_obs_score_shift_HI-Small.tex",
     "How the score moves. The one finding here (bank 012's score drifts upward) fits in a sentence."),
    ("table_obs_detection_all_HI-Small.tex, table_obs_detection_degree3_HI-Small.tex", "The numbers behind the detection figure, "
     "with positives; the degree >= 3 variant removes clients with too few neighbours to have a second-order neighbourhood."),
    ("table_obs_alerts_HI-Small.tex, table_obs_queue_HI-Small.tex", "Alert volume at score > 0.5 and precision at K=100 beside "
     "ties@K, which says whether the top K is uniquely defined."),
]
display(pd.DataFrame(HELD_APPX, columns=["file", "why it was held back"]))

---
## Before these go into the manuscript

* **State what hub removal does to the evaluated population.** The hub rows are scored on the accounts that remain after the hubs
  are deleted, and the deleted hubs include accounts with a laundering transaction (the numbers above). Say so beside the
  hub rows; the note under the ablation table already does.
* **Say what the sweep covers.** It runs on HI-Small only, scores the GARG-AML score alone and fits no model; LI-Large appears only in
  the pattern-splitting numbers.
* **Say what a runtime is.** One wall-clock run per setting on a shared node, with no split between the Louvain step and
  the scoring (the runs predate the per-stage timing); quote the large factors, not differences of tens of seconds.
* **Footnote the partitions.** Directed and undirected can be reduced by different Louvain partitions at one resolution, so a gap
  between them at a single setting is not purely a direction effect.
* **Appendix: state that Louvain is off on both sides** of the comparison, so its full-network column is not the main results'
  number, and that cells with few positives are shown but not interpreted.
* **Check the LaTeX widths** in your template: the tables are written for a wide float and the figures for full width; the snippets
  above use `figure*` for the wide ones.